# One Click NISS Statistical Report

Builds the NISS statistical record file, the transmittal file and the summary file for one reporting year.

**How to run**
1. Attach the default lakehouse (DMAnalytics) and make sure the policy lakehouse (ODS_STATS_TBLS_LH) is available.
2. Put the reference CSV files into the folder given by `files_root`.
3. Run all cells. To run from a Data Pipeline, pass any value of the parameters cell as a base parameter.

**Stages**
1. Policy extract: premium bearing coverage rows, exclusions, paired coverage merge, Origami proration.
2. Onset and offset rows: offset amounts get the coverage attributes of the referenced transaction.
3. Claim valuation: paid loss, reserve and allocated expense rows per segment.
4. Claim to policy mapping: claim rows are tied to policy coverages.
5. Zip codes: IBMi and Origami zip lookups.
6. Transformations: ASLOB, territory, NISS codes, exception codes, exposure, corrections.
7. Report files: NISS records, transmittal, summary, final report table and quality checks.

**Behaviour kept from the original notebook (please review with the business owner)**
- Claim rows with a null ReserveAmount are removed by the filter `ReserveAmount >= 0`. This removes all paid loss and expense rows from the final file. The switch `drop_claim_rows_with_null_reserve` keeps this legacy result when True. Set it to False to keep paid loss and expense rows.
- The exception code coverage list contains `ACC Death` (mixed case). The comparison is case sensitive, so `ACC DEATH` rows do not keep an exception code.
- The post year end correction uses a coverage mapping that differs from the main mapping for `RR` and `T&L` (see `correction_mapping_overrides`).
- The allocated expense query has no company code filter, as in the original.
- Rows without a claim code are not mapped to NISS codes and are not part of the output.
- Picking the first zip code per transaction and the first matching exception code per risk item is not deterministic when several candidates exist.

**Changes in behaviour that do not affect the file layout**
- The Constant table is no longer appended to on every run. Constants come from parameters, with an optional override table.
- Internal helper columns (`pair`, `Option1_clean`, `Deductible1_clean`) are no longer stored in the output tables.
- Reference files with duplicate keys are reduced to one row per key, with a warning, so that a left join cannot add rows.
- The final sort of the policy extract was removed because nothing depends on the order.

**Where to change things**
- Run values (year, valuation months, lakehouses, file locations, switches): the parameters cell.
- Business rules and reference data (mappings, overrides, exclusions): the `Rules` class in the configuration cell, or a JSON file referenced by `rules_override_path`. A field in the JSON file replaces the default of the same name.

In [ ]:
# Parameters cell. Every value below is a default that a Fabric pipeline or
# a notebook run can override. Lists are passed as comma separated strings.

# Reporting period
calendar_year = 2025                      # Policy book year and accident period being reported
call_year = 0                             # NISS call year; 0 means calendar_year + 1

# Source system filters
company_code = "14044"                    # Company code used in the claims fact tables
lob_code = "AC"                           # Policy line of business code
liability_eas_lob_codes = "193,194"       # Claim EAS LOB codes valued as liability
property_eas_lob_codes = "212"            # Claim EAS LOB codes valued as property
liability_valuation_month = "2026-03-01"  # First day of the liability valuation month
property_valuation_month = "2025-12-01"   # First day of the property valuation month
term_effective_after = "2023-06-30"       # Policy extract keeps TermEffectiveDate after this date
claim_term_effective_from = "2023-07-01"  # Claims keep policies with TermEffectiveDate on or after this date
reporting_states = "DE,IL,IN,KS,OH,OK,PA,VA"  # States that receive territory codes

# NISS layout constants
niss_company_number = "526"
niss_line_code = "1"
niss_commercial_indicator = "9"
niss_claimant_level_indicator = "C"
constants_table = ""                      # Optional table (ColumnName, ConstantValue) overriding the four values above

# Storage locations
ods_lakehouse = "ODS_STATS_TBLS_LH"       # Lakehouse that holds the policy tables
claims_lakehouse = "DMAnalytics"          # Lakehouse that holds the claims tables
output_lakehouse = ""                     # Empty means the default lakehouse of the notebook
files_root = "Files"                      # Folder that holds the reference CSV files
output_root = "Files"                     # Folder that receives the report files
territory_file_name = ""                  # Empty means Territory_code_<call_year>.csv
rules_override_path = ""                  # Optional JSON file that overrides business rules

# Output objects
table_onset_offset = "OnSet_And_Refrence_Offset"
table_claim_policy = "Claim_Policy_NewApproachData"
table_onset_offset_address = "OnSet_And_Refrence_Offset_Address"
table_final_transformed = "Final_Transformed_Data"
table_niss_final_report = "NISS_Final_Report"
output_transmittal = "TransmittalFILE"
output_summary = "SUMMARYFILE"
output_pre_final = "Pre_Final_Report_File"
output_niss_file = "First_NISS_FORMAT_FILE"

# Behaviour switches
drop_claim_rows_with_null_reserve = True  # True keeps the legacy filter ReserveAmount >= 0 (null rows are dropped)
drop_duplicate_rows = True                # Remove fully identical rows after policy enrichment
strict_lookup_uniqueness = False          # True fails the run when a reference file has duplicate keys
fail_on_missing_optional_files = False    # True fails the run when an optional correction file is missing
enable_validations = True                 # Row count and reference data checks
strict_quality_gate = False               # True fails the run when required output fields contain nulls
persist_intermediates = True              # Cache reused DataFrames in memory and disk

In [ ]:
import json
import logging
import re
import sys
import time
from contextlib import contextmanager
from dataclasses import dataclass, field, fields, replace
from datetime import date, datetime
from functools import reduce
from typing import Any, Dict, Iterable, List, Optional, Sequence, Tuple

from pyspark import StorageLevel
from pyspark.sql import Column, DataFrame, SparkSession, Window
from pyspark.sql import functions as F
from pyspark.sql import types as T

spark = SparkSession.builder.getOrCreate()



class _StdoutHandler(logging.StreamHandler):
    """Log handler that always writes to the current sys.stdout.

    Notebook runtimes can replace sys.stdout between cell runs. Resolving the
    stream at write time keeps the log visible after such a replacement.
    """

    def __init__(self) -> None:
        """Creates the handler."""
        super().__init__(sys.stdout)

    @property
    def stream(self):
        """Current standard output stream."""
        return sys.stdout

    @stream.setter
    def stream(self, value) -> None:
        """Ignores assignment because the stream is always sys.stdout."""


logger = logging.getLogger("niss_report")
if not logger.handlers:
    _handler = _StdoutHandler()
    _handler.setFormatter(
        logging.Formatter("%(asctime)s | %(levelname)-7s | %(message)s")
    )
    logger.addHandler(_handler)
logger.setLevel(logging.INFO)
logger.propagate = False

## Configuration

Parameters are resolved into `RunConfig`. Business rules live in `Rules`. Nothing below this cell contains a year, a policy number or a table name.

In [ ]:
PARAMETER_NAMES = (
    "calendar_year", "call_year", "company_code", "lob_code",
    "liability_eas_lob_codes", "property_eas_lob_codes",
    "liability_valuation_month", "property_valuation_month",
    "term_effective_after", "claim_term_effective_from", "reporting_states",
    "niss_company_number", "niss_line_code", "niss_commercial_indicator",
    "niss_claimant_level_indicator", "constants_table",
    "ods_lakehouse", "claims_lakehouse", "output_lakehouse", "files_root",
    "output_root", "territory_file_name", "rules_override_path",
    "table_onset_offset", "table_claim_policy", "table_onset_offset_address",
    "table_final_transformed", "table_niss_final_report",
    "output_transmittal", "output_summary", "output_pre_final", "output_niss_file",
    "drop_claim_rows_with_null_reserve", "drop_duplicate_rows",
    "strict_lookup_uniqueness", "fail_on_missing_optional_files",
    "enable_validations", "strict_quality_gate", "persist_intermediates",
)


def _as_bool(value: Any, name: str) -> bool:
    """Converts a pipeline parameter into a boolean.

    Args:
        value: Boolean or text such as "true" or "false".
        name: Parameter name used in the error message.

    Returns:
        The boolean value.

    Raises:
        ValueError: If the value cannot be interpreted as a boolean.
    """
    if isinstance(value, bool):
        return value
    text = str(value).strip().lower()
    if text in ("true", "1", "yes", "y"):
        return True
    if text in ("false", "0", "no", "n", ""):
        return False
    raise ValueError(f"Parameter '{name}' must be true or false, got {value!r}")


def _as_iso_date(value: Any, name: str) -> date:
    """Parses an ISO date parameter.

    Args:
        value: Text in the form YYYY-MM-DD.
        name: Parameter name used in the error message.

    Returns:
        The parsed date.

    Raises:
        ValueError: If the text is not a valid ISO date.
    """
    try:
        return datetime.strptime(str(value).strip(), "%Y-%m-%d").date()
    except ValueError as exc:
        raise ValueError(
            f"Parameter '{name}' must be an ISO date (YYYY-MM-DD), got {value!r}"
        ) from exc


def _as_list(value: Any, name: str) -> Tuple[str, ...]:
    """Splits a comma separated parameter into a tuple of non-empty values.

    Args:
        value: Comma separated text.
        name: Parameter name used in the error message.

    Returns:
        Tuple of trimmed values.

    Raises:
        ValueError: If the result is empty.
    """
    items = tuple(part.strip() for part in str(value).split(",") if part.strip())
    if not items:
        raise ValueError(f"Parameter '{name}' must contain at least one value")
    return items


@dataclass(frozen=True)
class Rules:
    """Business rules and reference data used by the pipeline.

    Every field has a default that reproduces the legacy notebook. Any field can
    be replaced by a JSON file referenced through the rules_override_path
    parameter. A field supplied in the JSON file replaces the default entirely.

    Rule lists use one shape: {"when": {column: value, ...}, "value": result}.
    A condition value can be a scalar, a list (membership) or null (is null).
    A list that contains null matches null values as well.
    """

    # Source systems
    ibmi_source: str = "IBMi"
    origami_source: str = "Origami"
    origami_policy_prefix: str = "GMCA"

    # Physical table and file names
    ods_tables: Dict[str, str] = field(default_factory=lambda: {
        "policy_detail": "policy_policy_detail_info",
        "product": "policy_Product_info",
        "risk_item": "policy_risk_item_info",
        "coverage": "policy_coverage_info",
        "question_answer": "policy_question_answer",
        "policy_address": "policy_policy_address",
        "party_address": "party_address",
    })
    claims_tables: Dict[str, str] = field(default_factory=lambda: {
        "claims_monthly": "FactClaimsMonthlyDataStore",
        "claims_fact": "FactClaims",
    })
    reference_files: Dict[str, str] = field(default_factory=lambda: {
        "duplicate_coverages": "DeleteDuplicateCoveragesFromSTATExtract.csv",
        "aslob_manual": "ASOLB_Manual_update.csv",
        "ba_plus_aslob": "BA PLUS-UpdateASLOB.csv",
        "missing_zip": "Missing_ZIP_Code.csv",
        "zip_fix": "ZIP_Further_fix.csv",
        "territory": "",
        "has_pip": "HasPIPVUWC_New.csv",
        "niss_mapping": "NISS_Column_Mapping-Updated.csv",
        "corrected_class": "Corrected_Class_code.csv",
        "cause_of_loss": "NissCauseofloss_df.csv",
        "exception_mapping": "Exception_code_Mapping.csv",
        "claim_indicator_classes": "ClassesWithClaimInd_9.csv",
        "datafix_offset": "DataFix-ForOffsetRecordsWithoutReferenceData.csv",
        "datafix_claims": "DataFix-Claims.csv",
    })
    optional_reference_files: List[str] = field(default_factory=lambda: [
        "duplicate_coverages", "aslob_manual", "ba_plus_aslob", "missing_zip",
        "zip_fix", "has_pip", "corrected_class", "datafix_offset", "datafix_claims",
    ])

    # Policy extract
    risk_type_vehicle: str = "Vehicle"
    risk_type_location: str = "Location"
    excluded_vehicle_class_codes: List[str] = field(
        default_factory=lambda: ["9410", "9437", "9582", "9235", "9670"]
    )
    exclude_policy_versions: List[Dict[str, Any]] = field(default_factory=lambda: [
        {"PolicyNumber": "GMCA100000107", "PolicyVersion": 1449},
        {"PolicyNumber": "GMCA100000080", "PolicyVersion": 4740},
    ])
    duplicate_coverage_keys: List[str] = field(default_factory=lambda: [
        "PolicyNumber", "PolicyVersion", "InTransactionId", "RiskItemId", "CoverageId",
    ])
    merge_coverage_pairs: List[List[str]] = field(default_factory=lambda: [
        ["BODINJLIAB", "BI"], ["COLLISION", "COLL"], ["OTC", "COMP"],
    ])
    prorata_question_code: str = "ProrataPercentage"
    unit_number_question_code: str = "PrintUnitNumber"
    location_question_code: str = "LocationNumber"
    primary_location_value: str = "1"
    origami_address_uses: List[str] = field(default_factory=lambda: [
        "Risk Address", "Garage Address-PolicyLocation",
        "Garage Address - Other", "Garage Address-Other",
    ])

    # Offset reference rows
    offset_parent_only_rules: List[Dict[str, Any]] = field(default_factory=lambda: [
        {"PolicyNumber": "GMCA001142870", "CoverageCode": "BROAD PIP",
         "ClaimCode": "BROAD PIP__NF"},
    ])

    # Claim extraction
    claim_coverage_mapping: List[List[str]] = field(default_factory=lambda: [
            ["ACC DEATH", "NF", "ACC DEATH_NF"],
            ["BA PLUS", "CM", "BA PLUS_TE"],
            ["BA PLUS", "RR", "BA PLUS_RR"],
            ["BA PLUS", "TL", "BA PLUS_T-L"],
            ["BA PLUS_RR", "CM", "BA PLUS_RR"],
            ["BA PLUS_T-L", "CM", "BA PLUS_T-L"],
            ["BI", "BI", "BODINJLIAB"],
            ["BI", "CBI", "BODINJLIAB"],
            ["BODINJLIAB", "BI", "BODINJLIAB"],
            ["BODINJLIAB", "CBI", "BODINJLIAB"],
            ["COLL", "CL", "COLLISION"],
            ["COLLISION", "CL", "COLLISION"],
            ["COMB-FPB", "FPB", "COMB-FPB"],
            ["COMP", "CM", "OTC"],
            ["FRMTRKPOLL", "PD", "FRMTRKPOLL_PRPDMGLIAB"],
            ["GK COLL", "CL", "GK COLL"],
            ["GK COMP", "CM", "GK COMP"],
            ["PIP LIMIT_INC LOSS", "NF", "PIP LIMIT_INC LOSS"],
            ["PIP LIMIT_ACC DEATH", "NF", "PIP LIMIT_ACC DEATH"],
            ["PIP LIMIT_FUN EXP", "NF", "PIP LIMIT_FUN EXP"],
            ["HIRED AUTO", "PD", "HIRED AUTO_PRPDMGLIAB"],
            ["HIRED PHYS", "CM", "HIRED PHYS_OTC"],
            ["INC LOSS", "NF", "INC LOSS"],
            ["MED EXP", "FPB", "MED EXP"],
            ["MED EXP", "NF", "MED EXP"],
            ["MEDPAYMENT", "MP", "MEDPAYMENT"],
            ["MP", "MP", "MEDPAYMENT"],
            ["OTC", "CM", "OTC"],
            ["PD", "CPD", "PRPDMGLIAB"],
            ["PD", "PD", "PRPDMGLIAB"],
            ["PIP LIMIT", "NF", "PIP LIMIT"],
            ["PIP LIMIT_MED EXP", "NF", "PIP LIMIT_MED EXP"],
            ["PRPDMGLIAB", "CPD", "PRPDMGLIAB"],
            ["PRPDMGLIAB", "PD", "PRPDMGLIAB"],
            ["RR", "CM", "RR"],
            ["RR", "RR", "RR"],
            ["T&L", "TL", "T-L"],
            ["T-L", "CM", "T-L"],
            ["UBI", "UBI", "UBI"],
            ["UBI NS", "UBI", "UBI"],
            ["UIM NS", "UBI", "UIM"],
            ["UIM ST", "UBI", "UBI"],
            ["UPD", "UPD", "UPD"],
        ])
    correction_mapping_overrides: List[List[str]] = field(default_factory=lambda: [
        ["RR", "RR", "BA PLUS_RR"],
        ["T&L", "TL", "BA PLUS_T-L"],
    ])
    segment_options: Dict[str, Dict[str, bool]] = field(default_factory=lambda: {
        "liability": {"roll_back_to_year_end": True, "restrict_loss_year_to_period": True},
        "property": {"roll_back_to_year_end": False, "restrict_loss_year_to_period": False},
    })
    claim_extension_collapse: Dict[str, Any] = field(default_factory=lambda: {
        "extensions": ["RR", "TL"], "to": "CM",
    })
    claim_ba_plus_rule: Dict[str, str] = field(default_factory=lambda: {
        "claim_coverage_name": "BA PLUS", "extension": "CM",
    })
    claim_med_exp_rule: Dict[str, str] = field(default_factory=lambda: {
        "derived_name": "MED EXP", "extension": "NF", "new_name": "PIP LIMIT_MED EXP",
    })
    derived_coverage_overrides: List[Dict[str, Any]] = field(default_factory=lambda: [
        {"when": {"PolicyNumber": 134028, "TransactionNumber": 26,
                  "ClaimNumber": 596598, "ClaimCoverageName": "MED EXP"},
         "value": "PIP LIMIT"},
    ])

    # Policy transformations
    ibmi_extension_by_coverage: Dict[str, str] = field(default_factory=lambda: {
        "UPD DED": "UPD", "PIP LIMIT": "NF", "BI": "CBI", "BA PLUS": "CM", "BROAD PIP": "BI",
    })
    ibmi_cbi_to_bi_coverages: List[str] = field(default_factory=lambda: [
        "FARMTR", "FRMTRKPOLL", "HIRED AUTO", "NONOWNAUTO", "DOC",
    ])
    origami_extension_by_coverage: Dict[str, str] = field(default_factory=lambda: {
        "PIP LIMIT": "NF",
    })
    ibmi_group_by_coverage: Dict[str, str] = field(default_factory=lambda: {
        "BA PLUS": "Property", "UPD DED": "Liability",
    })
    ibmi_aslob_by_coverage: Dict[str, int] = field(default_factory=lambda: {
        "BA PLUS": 212, "BROAD PIP": 194, "BI": 194, "BLNKT AI": 194, "DOC": 194,
        "FARMTR": 194, "FRMTRKPOLL": 194, "GK COLL": 212, "GK COMP": 212,
        "HIRED AUTO": 194, "HIRED PHYS": 212, "NONOWNAUTO": 194, "PIP LIMIT": 193,
        "UBI HIRED": 194, "UBI NONOWN": 194, "UIM HIRED": 194, "UIM NONOWN": 194,
        "UPD DED": 194,
    })
    origami_aslob_fallback: Dict[str, int] = field(default_factory=lambda: {
        "BODINJLIAB": 194, "COLLISION": 212, "UIM": 194, "COMB-FPB": 193,
        "MED EXP": 193, "OTC": 212, "UBI": 194,
    })
    ba_plus_aslob_value: int = 194
    zip_overrides: List[Dict[str, Any]] = field(default_factory=lambda: [
        {"when": {"InTransactionId": 2830107}, "value": 45356},
    ])
    rating_basis_question_code: str = "RatingBasis"
    has_pip_state: str = "PA"
    has_pip_extensions: List[str] = field(default_factory=lambda: ["NF", "FPB"])
    has_pip_rules: List[Dict[str, Any]] = field(default_factory=lambda: [
        {"when": {"CoverageCode": ["MED EXP", "INC LOSS", "ACC DEATH", "FUN EXP"],
                  "RB_TextValue": [3]}, "value": 1},
        {"when": {"CoverageCode": ["MED EXP", "ACC DEATH", "INC LOSS", "FUN EXP"],
                  "RB_TextValue": [0, 1, 2, 4]}, "value": 0},
        {"when": {"CoverageCode": ["COMB-FPB"], "RB_TextValue": [3]}, "value": 1},
        {"when": {"CoverageCode": ["COMB-FPB"], "RB_TextValue": [1, 2, 4]}, "value": 0},
        {"when": {"CoverageCode": ["EMB"], "RB_TextValue": [1, 3]}, "value": 1},
        {"when": {"CoverageCode": ["EMB"], "RB_TextValue": [2, 4]}, "value": 0},
        {"when": {"CoverageCode": ["BROAD PIP"]}, "value": 0},
    ])
    has_pip_default_pairs: List[str] = field(default_factory=lambda: [
        "MED EXP_NF", "MED EXP_FPB", "COMB-FPB_NF", "COMB-FPB_FPB", "EMB_NF", "EMB_FPB",
        "BROAD PIP_NF", "BROAD PIP_FPB", "ACC DEATH_NF", "ACC DEATH_FPB",
        "FUN EXP_NF", "FUN EXP_FPB", "INC LOSS_NF", "INC LOSS_FPB",
    ])
    post_extension_rules: List[Dict[str, Any]] = field(default_factory=lambda: [
        {"when": {"CoverageCode": "BA PLUS", "CoverageExtension": ["CBI", "BI"]}, "value": "CM"},
        {"when": {"CoverageCode": "MED EXP", "CoverageExtension": [None, ""]}, "value": "NF"},
    ])
    claim_code_separator: str = "__"
    option1_rules: List[Dict[str, Any]] = field(default_factory=lambda: [
        {"when": {"ClaimCode": ["UBI__UBI", "UIM__UBI"], "Option1": None,
                  "PrimaryRiskState": "PA"}, "value": "NS"},
    ])

    # NISS mapping
    valid_option1_values: List[str] = field(default_factory=lambda: ["NI", "NI&R", "NS", "ST"])
    deductible_claim_codes: List[str] = field(default_factory=lambda: ["OTC__CM", "COLLISION__CL"])
    deductible_state_claim_codes: List[Dict[str, str]] = field(default_factory=lambda: [
        {"claim_code": "PIP LIMIT__NF", "state": "DE"},
    ])
    niss_code_overrides: List[Dict[str, Any]] = field(default_factory=lambda: [
        {"when": {"ClaimCode": ["BA PLUS__CBI", "BA PLUS__BI"]},
         "set": {"NISS_Coverage_Code": 19, "NISS_Subline_Code": 0,
                 "NISS_Type_of_Loss": None, "NISS_Class_Codes": 9999}},
        {"when": {"PolicySource": "IBMi", "CoverageCode": "UBI HIRED"},
         "set": {"NISS_Coverage_Code": 201, "NISS_Subline_Code": 0,
                 "NISS_Type_of_Loss": 0, "NISS_Class_Codes": 9404}},
        {"when": {"PolicySource": "IBMi", "CoverageCode": "UBI NONOWN"},
         "set": {"NISS_Coverage_Code": 201, "NISS_Subline_Code": 0,
                 "NISS_Type_of_Loss": 0, "NISS_Class_Codes": 9408}},
        {"when": {"PolicySource": "IBMi", "CoverageCode": "UIM HIRED"},
         "set": {"NISS_Coverage_Code": 202, "NISS_Subline_Code": 0,
                 "NISS_Type_of_Loss": 0, "NISS_Class_Codes": 9404}},
        {"when": {"PolicySource": "IBMi", "CoverageCode": "UIM NONOWN"},
         "set": {"NISS_Coverage_Code": 202, "NISS_Subline_Code": 0,
                 "NISS_Type_of_Loss": 0, "NISS_Class_Codes": 9408}},
    ])
    vehicle_class_placeholder: str = "Vehicle Class Code"
    niss_class_code_length: int = 4

    # Exception codes
    exception_anchor_coverage: str = "MED EXP"
    exception_anchor_extensions: List[str] = field(default_factory=lambda: ["NF", "FPB"])
    exception_anchor_state: str = "PA"
    exception_all_of: List[List[str]] = field(default_factory=lambda: [
        ["MED EXP", "Limit1"], ["ACC DEATH", "Limit1"], ["FUN EXP", "Limit1"],
    ])
    exception_any_of: List[List[str]] = field(default_factory=lambda: [
        ["INC LOSS", "Limit1"], ["INC LOSS", "Limit2"],
    ])
    exception_by_limit: Dict[str, Dict[str, Any]] = field(default_factory=lambda: {
        "EMB": {"limits": [[100000, "01"], [300000, "02"], [500000, "03"], [1000000, "04"]],
                "default": None},
        "COMB-FPB": {"limits": [[12500, "06"], [17500, "01"], [50000, "02"], [100000, "03"],
                                [177500, "07"], [200000, "04"], [277500, "05"]],
                     "default": "09"},
    })
    # The value "ACC Death" is kept exactly as in the legacy notebook. Comparison is
    # case sensitive, so rows with coverage code "ACC DEATH" do not keep a code here.
    exception_applicable_coverages: List[str] = field(default_factory=lambda: [
        "COMB-FPB", "EMB", "MED EXP", "ACC Death", "FUN EXP",
    ])
    exception_applicable_with_extension: List[Dict[str, str]] = field(default_factory=lambda: [
        {"CoverageCode": "INC LOSS", "CoverageExtension": "NF"},
    ])
    exception_code_overrides: List[Dict[str, Any]] = field(default_factory=lambda: [
        {"when": {"InTransactionId": 3203888, "RiskItemId": 2174291,
                  "PolicyNumber": "GMCA100000183", "PolicyVersion": 1824,
                  "CoverageId": 17000057, "ClaimCode": "INC LOSS__NF"}, "value": "53"},
        {"when": {"InTransactionId": 3276779, "RiskItemId": 2216881,
                  "PolicyNumber": "GMCA001138996", "PolicyVersion": 2215,
                  "CoverageId": 17390445, "ClaimCode": "INC LOSS__NF"}, "value": "53"},
        {"when": {"InTransactionId": 156722, "RiskItemId": 92935,
                  "PolicyNumber": "135096", "PolicyVersion": 12,
                  "CoverageId": 653539, "ClaimCode": "INC LOSS__NF"}, "value": "52"},
    ])
    final_exception_overrides: List[Dict[str, Any]] = field(default_factory=lambda: [
        {"when": {"PrimaryRiskState": "37", "CoverageCode": "BROAD PIP"}, "value": "49"},
    ])

    # Exposure, state codes and corrections
    prorata_default_value: int = 1
    exposure_months: int = 12
    state_code_map: Dict[str, str] = field(default_factory=lambda: {
        "DE": "07", "IL": "12", "IN": "13", "KS": "15",
        "OH": "34", "OK": "35", "PA": "37", "VA": "45",
    })
    claim_coverage_code_overrides: List[Dict[str, Any]] = field(default_factory=lambda: [
        {"when": {"ClaimNumber": 633662, "ClaimCoverageName": "BA PLUS_RR"},
         "value": "BA PLUS"},
    ])

    # Report layout adjustments
    exposure_blank_class_codes: List[str] = field(default_factory=lambda: [
        "608000", "604000", "580000", "600100", "600300", "600500", "600900",
        "648000", "649000", "780000", "790000", "708000", "940900", "949500",
        "940600", "940500", "940400", "999900", "940800", "941100", "941400",
        "941200", "946400",
    ])
    quality_required_columns: List[str] = field(default_factory=lambda: [
        "StateCode", "NISSCoverageCode", "AnnualStatementLineOfBusinessCode",
    ])

    def __post_init__(self) -> None:
        """Validates the shape of the rule data.

        Raises:
            ValueError: If the coverage mapping has duplicate keys or malformed rows.
        """
        keys = [(row[0], row[1]) for row in self.claim_coverage_mapping]
        if len(keys) != len(set(keys)):
            raise ValueError("claim_coverage_mapping contains duplicate (name, extension) keys")
        for pair in self.merge_coverage_pairs:
            if len(pair) != 2:
                raise ValueError(f"merge_coverage_pairs entries need two codes, got {pair!r}")
        for row in list(self.claim_coverage_mapping) + list(self.correction_mapping_overrides):
            if len(row) != 3:
                raise ValueError(f"Coverage mapping rows need three values, got {row!r}")


def load_rules(path: str) -> Rules:
    """Builds the rule set, applying an optional JSON override file.

    Args:
        path: Lakehouse path of a JSON file. An empty string uses the defaults.

    Returns:
        The rule set.

    Raises:
        ValueError: If the JSON contains a field that is not a rule.
    """
    if not path:
        return Rules()
    rows = spark.read.option("wholetext", "true").text(path).collect()
    overrides = json.loads("\n".join(row[0] for row in rows))
    known = {item.name for item in fields(Rules)}
    unknown = sorted(set(overrides) - known)
    if unknown:
        raise ValueError(f"Unknown rule names in {path}: {unknown}")
    logger.info("Rules overridden from %s: %s", path, sorted(overrides))
    return replace(Rules(), **overrides)


@dataclass(frozen=True)
class ClaimSegment:
    """Valuation settings for one group of claim lines of business.

    Attributes:
        name: Segment name, either liability or property.
        eas_lob_codes: EAS LOB codes that belong to the segment.
        valuation_month: First day of the month used for the reserve valuation.
        max_loss_year_exclusive: Loss years must be below this value. None disables the filter.
        roll_back_to_year_end: True adds payments made after year end back to the reserve.
    """

    name: str
    eas_lob_codes: Tuple[str, ...]
    valuation_month: date
    max_loss_year_exclusive: Optional[int]
    roll_back_to_year_end: bool

    @property
    def incurred_end_exclusive(self) -> date:
        """First day of the month that follows the valuation month."""
        month = self.valuation_month.month
        year = self.valuation_month.year + month // 12
        return date(year, month % 12 + 1, 1)


@dataclass(frozen=True)
class RunConfig:
    """Resolved run settings derived from the notebook parameters.

    Attributes:
        calendar_year: Reporting year.
        call_year: NISS call year.
        company_code: Company code in the claims tables.
        lob_code: Policy line of business code.
        liability: Liability claim segment.
        property_segment: Property claim segment.
        term_effective_after: Policy extract lower bound, exclusive.
        claim_term_effective_from: Claim filter lower bound, inclusive.
        reporting_states: State abbreviations that receive territory codes.
        layout_constants: Constant values written into the NISS records.
        constants_table: Optional table that overrides the layout constants.
        ods_lakehouse: Lakehouse that holds the policy tables.
        claims_lakehouse: Lakehouse that holds the claims tables.
        output_lakehouse: Lakehouse prefix for output tables, empty for the default.
        files_root: Folder that holds the reference CSV files.
        output_root: Folder that receives the report files.
        territory_file_name: Territory file name, empty for the default name.
        table_onset_offset: Checkpoint table with onset and offset rows.
        table_claim_policy: Checkpoint table with claim rows mapped to policy rows.
        table_onset_offset_address: Checkpoint table that includes zip codes.
        table_final_transformed: Table with the fully transformed rows.
        table_niss_final_report: Table with the final report layout.
        output_transmittal: Folder name of the transmittal file.
        output_summary: Folder name of the summary file.
        output_pre_final: Folder name of the pre final diagnostic file.
        output_niss_file: Folder name of the NISS record file.
        drop_claim_rows_with_null_reserve: Legacy reserve filter switch.
        drop_duplicate_rows: Remove fully identical rows after enrichment.
        strict_lookup_uniqueness: Fail when a reference file has duplicate keys.
        fail_on_missing_optional_files: Fail when an optional file is missing.
        enable_validations: Run row count and reference data checks.
        strict_quality_gate: Fail when required output fields contain nulls.
        persist_intermediates: Cache reused DataFrames.
    """

    calendar_year: int
    call_year: int
    company_code: str
    lob_code: str
    liability: ClaimSegment
    property_segment: ClaimSegment
    term_effective_after: str
    claim_term_effective_from: str
    reporting_states: Tuple[str, ...]
    layout_constants: Dict[str, str]
    constants_table: str
    ods_lakehouse: str
    claims_lakehouse: str
    output_lakehouse: str
    files_root: str
    output_root: str
    territory_file_name: str
    table_onset_offset: str
    table_claim_policy: str
    table_onset_offset_address: str
    table_final_transformed: str
    table_niss_final_report: str
    output_transmittal: str
    output_summary: str
    output_pre_final: str
    output_niss_file: str
    drop_claim_rows_with_null_reserve: bool
    drop_duplicate_rows: bool
    strict_lookup_uniqueness: bool
    fail_on_missing_optional_files: bool
    enable_validations: bool
    strict_quality_gate: bool
    persist_intermediates: bool

    @property
    def period_start(self) -> date:
        """First day of the reporting year."""
        return date(self.calendar_year, 1, 1)

    @property
    def period_end(self) -> date:
        """Last day of the reporting year."""
        return date(self.calendar_year, 12, 31)

    @property
    def next_year_start(self) -> date:
        """First day of the year after the reporting year."""
        return date(self.calendar_year + 1, 1, 1)

    @property
    def two_digit_calendar_year(self) -> str:
        """Last two digits of the reporting year."""
        return str(self.calendar_year)[-2:]

    @property
    def two_digit_call_year(self) -> str:
        """Last two digits of the call year."""
        return str(self.call_year)[-2:]

    @property
    def segments(self) -> Tuple[ClaimSegment, ClaimSegment]:
        """Both claim segments in processing order."""
        return (self.liability, self.property_segment)


def build_run_config(rules: Rules) -> RunConfig:
    """Builds the run configuration from the notebook parameters.

    Args:
        rules: Rule set, used for the segment options.

    Returns:
        The validated configuration.

    Raises:
        NameError: If a parameter is missing from the parameters cell.
        ValueError: If a parameter has an invalid value.
    """
    values = globals()
    missing = [name for name in PARAMETER_NAMES if name not in values]
    if missing:
        raise NameError(f"Parameters missing from the parameters cell: {missing}")

    def text(name: str) -> str:
        """Reads a text parameter."""
        return str(values[name]).strip()

    def flag(name: str) -> bool:
        """Reads a boolean parameter."""
        return _as_bool(values[name], name)

    calendar_year = int(values["calendar_year"])
    call_year = int(values["call_year"]) or calendar_year + 1
    if not 1990 <= calendar_year <= 2100:
        raise ValueError(f"calendar_year looks wrong: {calendar_year}")

    def make_segment(name: str, codes_param: str, month_param: str) -> ClaimSegment:
        """Builds one claim segment from its parameters."""
        options = rules.segment_options[name]
        month = _as_iso_date(values[month_param], month_param)
        if month.day != 1:
            raise ValueError(f"{month_param} must be the first day of a month")
        return ClaimSegment(
            name=name,
            eas_lob_codes=_as_list(values[codes_param], codes_param),
            valuation_month=month,
            max_loss_year_exclusive=(
                calendar_year + 1 if options["restrict_loss_year_to_period"] else None
            ),
            roll_back_to_year_end=bool(options["roll_back_to_year_end"]),
        )

    return RunConfig(
        calendar_year=calendar_year,
        call_year=call_year,
        company_code=text("company_code"),
        lob_code=text("lob_code"),
        liability=make_segment(
            "liability", "liability_eas_lob_codes", "liability_valuation_month"
        ),
        property_segment=make_segment(
            "property", "property_eas_lob_codes", "property_valuation_month"
        ),
        term_effective_after=_as_iso_date(
            values["term_effective_after"], "term_effective_after"
        ).isoformat(),
        claim_term_effective_from=_as_iso_date(
            values["claim_term_effective_from"], "claim_term_effective_from"
        ).isoformat(),
        reporting_states=_as_list(values["reporting_states"], "reporting_states"),
        layout_constants={
            "CompanyNumber": text("niss_company_number"),
            "LineCode": text("niss_line_code"),
            "CommercialIndicatorCode": text("niss_commercial_indicator"),
            "ClaimantLevelIndicator": text("niss_claimant_level_indicator"),
        },
        constants_table=text("constants_table"),
        ods_lakehouse=text("ods_lakehouse"),
        claims_lakehouse=text("claims_lakehouse"),
        output_lakehouse=text("output_lakehouse"),
        files_root=text("files_root"),
        output_root=text("output_root"),
        territory_file_name=text("territory_file_name"),
        table_onset_offset=text("table_onset_offset"),
        table_claim_policy=text("table_claim_policy"),
        table_onset_offset_address=text("table_onset_offset_address"),
        table_final_transformed=text("table_final_transformed"),
        table_niss_final_report=text("table_niss_final_report"),
        output_transmittal=text("output_transmittal"),
        output_summary=text("output_summary"),
        output_pre_final=text("output_pre_final"),
        output_niss_file=text("output_niss_file"),
        drop_claim_rows_with_null_reserve=flag("drop_claim_rows_with_null_reserve"),
        drop_duplicate_rows=flag("drop_duplicate_rows"),
        strict_lookup_uniqueness=flag("strict_lookup_uniqueness"),
        fail_on_missing_optional_files=flag("fail_on_missing_optional_files"),
        enable_validations=flag("enable_validations"),
        strict_quality_gate=flag("strict_quality_gate"),
        persist_intermediates=flag("persist_intermediates"),
    )

## Shared helpers

Logging, stage timing, cache handling, rule engine, lookup protection and Delta writes.

In [ ]:
RUN_METRICS: List[Dict[str, Any]] = []


@contextmanager
def stage(name: str):
    """Times a pipeline stage and records the outcome.

    Args:
        name: Stage name used in the log and in the run summary.

    Yields:
        Nothing. Exceptions are logged and re-raised.
    """
    started = time.time()
    status = "ok"
    logger.info("START %s", name)
    try:
        yield
    except BaseException:
        status = "failed"
        logger.error("FAILED %s", name)
        raise
    finally:
        seconds = round(time.time() - started, 1)
        RUN_METRICS.append({"stage": name, "status": status, "seconds": seconds})
        logger.info("END   %s | %s | %.1f s", name, status, seconds)


class CacheRegistry:
    """Tracks persisted DataFrames so every one of them is released."""

    def __init__(self, enabled: bool = True) -> None:
        """Creates the registry.

        Args:
            enabled: When False, keep() returns the DataFrame unchanged.
        """
        self.enabled = enabled
        self._frames: List[Tuple[str, DataFrame]] = []

    def keep(self, df: DataFrame, label: str) -> DataFrame:
        """Persists a DataFrame in memory and disk.

        Args:
            df: DataFrame that is reused by several later steps.
            label: Name used in log messages.

        Returns:
            The same DataFrame.
        """
        if not self.enabled:
            return df
        df.persist(StorageLevel.MEMORY_AND_DISK)
        self._frames.append((label, df))
        return df

    def release_all(self) -> None:
        """Unpersists every DataFrame that was kept."""
        for label, frame in reversed(self._frames):
            try:
                frame.unpersist()
            except Exception as exc:  # cache release must never hide the real error
                logger.warning("Could not release cache %s: %s", label, exc)
        if self._frames:
            logger.info("Released %d cached DataFrames", len(self._frames))
        self._frames.clear()


def qualify(lakehouse: str, table: str) -> str:
    """Prefixes a table name with a lakehouse name when one is given.

    Args:
        lakehouse: Lakehouse name or an empty string.
        table: Table name.

    Returns:
        The qualified table name.
    """
    return f"{lakehouse}.{table}" if lakehouse else table


def sql_literal(value: Any) -> str:
    """Quotes a value as a SQL string literal.

    Args:
        value: Value to quote.

    Returns:
        The quoted literal with embedded quotes escaped.
    """
    return "'" + str(value).replace("'", "''") + "'"


def sql_list(values: Iterable[Any]) -> str:
    """Formats values as a comma separated list of SQL string literals."""
    return ", ".join(sql_literal(value) for value in values)


@dataclass
class Context:
    """Objects shared by all pipeline stages.

    Attributes:
        cfg: Resolved run configuration.
        rules: Business rules.
        cache: Registry of persisted DataFrames.
    """

    cfg: RunConfig
    rules: Rules
    cache: CacheRegistry

    def ods(self, key: str) -> str:
        """Returns the qualified name of a policy table."""
        return qualify(self.cfg.ods_lakehouse, self.rules.ods_tables[key])

    def claims(self, key: str) -> str:
        """Returns the qualified name of a claims table."""
        return qualify(self.cfg.claims_lakehouse, self.rules.claims_tables[key])

    def out(self, table: str) -> str:
        """Returns the qualified name of an output table."""
        return qualify(self.cfg.output_lakehouse, table)

    def read_ods(self, key: str) -> DataFrame:
        """Reads a policy table."""
        return spark.table(self.ods(key))

    def read_claims(self, key: str) -> DataFrame:
        """Reads a claims table."""
        return spark.table(self.claims(key))

    def file_path(self, file_name: str) -> str:
        """Returns the path of a reference file."""
        return f"{self.cfg.files_root}/{file_name}"

    def output_path(self, folder: str) -> str:
        """Returns the path of an output folder."""
        return f"{self.cfg.output_root}/{folder}"


def _is_missing_path_error(error: Exception) -> bool:
    """Tells whether an exception means that a file path does not exist."""
    message = str(error)
    return any(
        marker in message
        for marker in ("Path does not exist", "PATH_NOT_FOUND", "FileNotFoundException")
    )


def read_reference_csv(ctx: Context, key: str) -> Optional[DataFrame]:
    """Reads a reference CSV file with a header and inferred types.

    Args:
        ctx: Pipeline context.
        key: Key of the file in Rules.reference_files.

    Returns:
        The DataFrame, or None when an optional file is missing and the run is
        allowed to continue without it.

    Raises:
        Exception: If a required file is missing or cannot be read.
    """
    file_name = ctx.rules.reference_files[key]
    if key == "territory" and not file_name:
        file_name = ctx.cfg.territory_file_name or f"Territory_code_{ctx.cfg.call_year}.csv"
    path = ctx.file_path(file_name)
    try:
        df = (
            spark.read.option("header", "true").option("inferSchema", "true").csv(path)
        )
        df.columns  # forces schema resolution so that a missing file fails here
    except Exception as exc:
        optional = key in ctx.rules.optional_reference_files
        if _is_missing_path_error(exc) and optional and not ctx.cfg.fail_on_missing_optional_files:
            logger.warning("Optional reference file not found, step skipped: %s", path)
            return None
        raise
    logger.info("Read reference file %s", path)
    return df


def require_columns(df: DataFrame, columns: Sequence[str], label: str) -> None:
    """Checks that a DataFrame has the expected columns.

    Args:
        df: DataFrame to check.
        columns: Column names that must exist.
        label: Name used in the error message.

    Raises:
        ValueError: If a column is missing.
    """
    missing = [name for name in columns if name not in df.columns]
    if missing:
        raise ValueError(f"{label} is missing columns {missing}; found {df.columns}")


def prepare_lookup(ctx: Context, df: DataFrame, keys: Sequence[str], label: str) -> DataFrame:
    """Guarantees at most one row per key so that a left join cannot add rows.

    Args:
        ctx: Pipeline context.
        df: Lookup DataFrame.
        keys: Join key columns.
        label: Name used in log messages.

    Returns:
        The lookup, de-duplicated on the keys.

    Raises:
        ValueError: If duplicates exist and strict_lookup_uniqueness is on.
    """
    require_columns(df, keys, label)
    if ctx.cfg.enable_validations:
        duplicated = (
            df.groupBy(*keys).count().filter(F.col("count") > 1).count()
        )
        if duplicated:
            message = f"{label} has {duplicated} duplicated key values on {list(keys)}"
            if ctx.cfg.strict_lookup_uniqueness:
                raise ValueError(message)
            logger.warning("%s; one row per key is kept", message)
    return df.dropDuplicates(list(keys))


def restrict_to_keys(
    df: DataFrame, keys: DataFrame, key_columns: Sequence[str], broadcast: bool = True
) -> DataFrame:
    """Keeps only the rows of a large table whose keys occur in a small key set.

    Args:
        df: Large DataFrame to reduce.
        keys: DataFrame that contains the key columns.
        key_columns: Names of the key columns, present in both DataFrames.
        broadcast: Broadcast the key set to every executor.

    Returns:
        The reduced DataFrame with its original columns.
    """
    key_frame = keys.select(*key_columns).distinct()
    if broadcast:
        key_frame = F.broadcast(key_frame)
    return df.join(key_frame, list(key_columns), "left_semi")


def first_row_per_key(
    df: DataFrame, partition_columns: Sequence[str], order_columns: Sequence[Any]
) -> DataFrame:
    """Keeps the first row of every key after sorting.

    Args:
        df: Input DataFrame.
        partition_columns: Columns that define a key.
        order_columns: Column names or Column objects that define the sort order.

    Returns:
        One row per key.
    """
    window = Window.partitionBy(*partition_columns).orderBy(*order_columns)
    return (
        df.withColumn("_row_rank", F.row_number().over(window))
        .filter(F.col("_row_rank") == 1)
        .drop("_row_rank")
    )


def union_all(frames: Sequence[DataFrame]) -> DataFrame:
    """Combines DataFrames by column name."""
    return reduce(lambda left, right: left.unionByName(right), frames)


def condition_from_dict(spec: Dict[str, Any]) -> Column:
    """Builds a boolean column from a rule condition dictionary.

    Args:
        spec: Mapping of column name to expected value. A scalar means equality,
            a list means membership, null means is null. A list that contains
            null also matches null values.

    Returns:
        All conditions combined with AND.

    Raises:
        ValueError: If the dictionary is empty.
    """
    if not spec:
        raise ValueError("A rule needs at least one condition")
    parts: List[Column] = []
    for name, expected in spec.items():
        column = F.col(name)
        if isinstance(expected, (list, tuple)):
            present = [item for item in expected if item is not None]
            expression = column.isin(*present) if present else F.lit(False)
            if len(present) != len(expected):
                expression = column.isNull() | expression
        elif expected is None:
            expression = column.isNull()
        else:
            expression = column == F.lit(expected)
        parts.append(expression)
    return reduce(lambda left, right: left & right, parts)


def when_chain(rules: Sequence[Dict[str, Any]], otherwise: Column) -> Column:
    """Builds a CASE expression from rules. The first matching rule wins.

    Args:
        rules: Rules shaped as {"when": {...}, "value": result}.
        otherwise: Expression used when no rule matches.

    Returns:
        The CASE expression, or the fallback when there are no rules.
    """
    expression: Optional[Column] = None
    for rule in rules:
        condition = condition_from_dict(rule["when"])
        value = F.lit(rule["value"])
        expression = F.when(condition, value) if expression is None else expression.when(condition, value)
    return otherwise if expression is None else expression.otherwise(otherwise)


def apply_value_rules(df: DataFrame, target: str, rules: Sequence[Dict[str, Any]]) -> DataFrame:
    """Overwrites a column using rules. Non matching rows keep their value.

    Args:
        df: Input DataFrame.
        target: Column to update.
        rules: Rules shaped as {"when": {...}, "value": result}.

    Returns:
        The updated DataFrame.
    """
    if not rules:
        return df
    return df.withColumn(target, when_chain(rules, F.col(target)))


def mapping_rules(
    base_conditions: Dict[str, Any], key_column: str, mapping: Dict[str, Any]
) -> List[Dict[str, Any]]:
    """Expands a code to value dictionary into rule dictionaries.

    Args:
        base_conditions: Conditions shared by every rule.
        key_column: Column compared with each dictionary key.
        mapping: Dictionary of expected value to result value.

    Returns:
        Rules shaped as {"when": {...}, "value": result}.
    """
    return [
        {"when": {**base_conditions, key_column: key}, "value": value}
        for key, value in mapping.items()
    ]


def apply_multi_column_rules(df: DataFrame, rules: Sequence[Dict[str, Any]]) -> DataFrame:
    """Overwrites several columns using rules shaped as {"when": {...}, "set": {...}}.

    The first matching rule wins for each column. Columns that a rule does not
    set keep their value.

    Args:
        df: Input DataFrame.
        rules: Rules with a set dictionary of column to value.

    Returns:
        The updated DataFrame.
    """
    targets: List[str] = []
    for rule in rules:
        for name in rule["set"]:
            if name not in targets:
                targets.append(name)
    updates = {}
    for name in targets:
        per_column = [
            {"when": rule["when"], "value": rule["set"][name]}
            for rule in rules
            if name in rule["set"]
        ]
        updates[name] = when_chain(per_column, F.col(name))
    return df.withColumns(updates) if updates else df


def null_safe_equal(left: str, right: str) -> Column:
    """Null safe equality between two columns."""
    return F.col(left).eqNullSafe(F.col(right))


def write_delta_table(ctx: Context, df: DataFrame, table_name: str) -> DataFrame:
    """Writes a Delta table, replacing data and schema, and returns it re-read.

    Columns of void type cannot be stored in Delta, so they are stored as string.

    Args:
        ctx: Pipeline context.
        df: DataFrame to store.
        table_name: Target table name.

    Returns:
        The stored table read back. Reading it back cuts the query lineage.
    """
    safe = df
    for item in df.schema.fields:
        if isinstance(item.dataType, T.NullType):
            safe = safe.withColumn(item.name, F.col(item.name).cast("string"))
    (
        safe.write.format("delta")
        .mode("overwrite")
        .option("overwriteSchema", "true")
        .saveAsTable(table_name)
    )
    stored = spark.table(table_name)
    if ctx.cfg.enable_validations:
        logger.info("Stored %s rows in %s", f"{stored.count():,}", table_name)
    return stored


def write_csv_folder(df: DataFrame, path: str, header: bool) -> None:
    """Writes a DataFrame as one CSV part file inside a folder.

    Args:
        df: DataFrame to write.
        path: Output folder.
        header: Whether to write the column names.
    """
    (
        df.coalesce(1)
        .write.mode("overwrite")
        .option("header", "true" if header else "false")
        .csv(path)
    )
    logger.info("Wrote CSV folder %s", path)

## Stage 1: Policy extract

In [ ]:
POLICY_KEY_COLUMNS = [
    "PolicyNumber", "PolicyVersion", "InTransactionId", "ReferencePolicyVersion",
    "PolicySource", "BookDate", "ProrataPercent", "PrimaryRiskState",
]

COVERAGE_COLUMNS = [
    "CoverageId", "ParentCoverageId", "RiskItemId", "ProductId", "CoverageCode",
    "CoverageExtension", "CoverageGroup", "OnsetAmount", "OffsetAmount", "Limit1",
    "Limit2", "ASLOB", "Option1", "Exposure", "Deductible1", "ClaimCode",
    "CoverageInfoGUID",
]


def _policy_projection(with_risk_columns: bool) -> List[Column]:
    """Builds the output column list of the policy extract.

    Args:
        with_risk_columns: True takes the risk item columns from the data. False
            writes typed nulls, used for coverages that are not tied to a risk item.

    Returns:
        Columns in the order used by every extract block.
    """
    if with_risk_columns:
        risk = [F.col("RiskItemType"), F.col("UnitNumber"), F.col("ClassCode")]
    else:
        risk = [
            F.lit(None).cast("string").alias("RiskItemType"),
            F.lit(None).cast("int").alias("UnitNumber"),
            F.lit(None).cast("string").alias("ClassCode"),
        ]
    leading = ["PolicyNumber", "PolicyVersion", "InTransactionId", "ReferencePolicyVersion",
               "PolicySource", "BookDate", "ProrataPercent", "LOBCode"]
    return (
        [F.col(name) for name in leading]
        + risk
        + [F.col("PrimaryRiskState")]
        + [F.col(name) for name in COVERAGE_COLUMNS]
    )


def extract_policy_coverages(ctx: Context) -> DataFrame:
    """Extracts premium bearing coverage rows for the reporting year.

    Four kinds of coverage rows are combined: vehicle risk coverages, location
    risk coverages, product level coverages and coverages with neither a risk
    item nor a product. Only rows with a non-zero onset or offset amount remain.

    Args:
        ctx: Pipeline context.

    Returns:
        Distinct coverage rows with the 29 policy columns.
    """
    cfg, rules = ctx.cfg, ctx.rules

    policy_header = (
        ctx.read_ods("policy_detail")
        .filter(
            (F.col("BookDate") >= F.lit(cfg.period_start.isoformat()))
            & (F.col("BookDate") < F.lit(cfg.next_year_start.isoformat()))
            & (F.col("TermEffectiveDate") > F.lit(cfg.term_effective_after))
        )
        .select(*POLICY_KEY_COLUMNS)
    )
    product = (
        ctx.read_ods("product")
        .filter((F.col("LOBCode") == cfg.lob_code) & F.col("ParentProductID").isNull())
        .select("InTransactionId", "LOBCode", F.col("ProductId").alias("PolicyProductId"))
    )
    policy_base = ctx.cache.keep(
        policy_header.join(product, "InTransactionId", "inner"), "policy_base"
    )
    transaction_ids = policy_base.select("InTransactionId")

    has_amount = (
        (F.col("OnsetAmount").isNotNull() & (F.col("OnsetAmount") != 0))
        | (F.col("OffsetAmount").isNotNull() & (F.col("OffsetAmount") != 0))
    )
    coverage = (
        restrict_to_keys(ctx.read_ods("coverage"), transaction_ids, ["InTransactionId"])
        .filter(has_amount)
        .select("InTransactionId", *COVERAGE_COLUMNS)
    )

    vehicle_ok = F.col("RiskItemType") == rules.risk_type_vehicle
    if rules.excluded_vehicle_class_codes:
        vehicle_ok = vehicle_ok & ~F.col("ClassCode").isin(*rules.excluded_vehicle_class_codes)
    risk_items = (
        restrict_to_keys(ctx.read_ods("risk_item"), transaction_ids, ["InTransactionId"])
        .filter(vehicle_ok | (F.col("RiskItemType") == rules.risk_type_location))
        .select("InTransactionId", "RiskItemId", "RiskItemType", "UnitNumber", "ClassCode")
    )

    risk_level = (
        policy_base.join(risk_items, "InTransactionId")
        .join(coverage.filter(F.col("ProductId").isNull()), ["InTransactionId", "RiskItemId"])
        .select(*_policy_projection(True))
    )
    product_level = (
        policy_base.join(coverage.filter(F.col("RiskItemId").isNull()), "InTransactionId")
        .filter(F.col("ProductId") == F.col("PolicyProductId"))
        .select(*_policy_projection(False))
    )
    policy_level = (
        policy_base.join(
            coverage.filter(F.col("ProductId").isNull() & F.col("RiskItemId").isNull()),
            "InTransactionId",
        )
        .select(*_policy_projection(False))
    )
    return union_all([risk_level, product_level, policy_level]).distinct()


def exclude_policy_versions(ctx: Context, df: DataFrame) -> DataFrame:
    """Removes policy versions listed in Rules.exclude_policy_versions.

    Args:
        ctx: Pipeline context.
        df: Policy coverage rows.

    Returns:
        Rows whose (PolicyNumber, PolicyVersion) is not excluded.
    """
    pairs = [(item["PolicyNumber"], item["PolicyVersion"]) for item in ctx.rules.exclude_policy_versions]
    if not pairs:
        return df
    exclusions = spark.createDataFrame(pairs, ["PolicyNumber", "PolicyVersion"])
    return df.join(F.broadcast(exclusions), ["PolicyNumber", "PolicyVersion"], "left_anti")


def exclude_duplicate_coverages(ctx: Context, df: DataFrame) -> DataFrame:
    """Removes coverage rows listed in the duplicate coverage reference file.

    Args:
        ctx: Pipeline context.
        df: Policy coverage rows.

    Returns:
        Rows that are not listed in the file. Unchanged when the file is missing.
    """
    listed = read_reference_csv(ctx, "duplicate_coverages")
    if listed is None:
        return df
    keys = ctx.rules.duplicate_coverage_keys
    require_columns(listed, keys, "duplicate coverage file")
    return df.join(F.broadcast(listed.select(*keys)), keys, "left_anti")


def merge_paired_coverages(df: DataFrame, pairs: Sequence[Sequence[str]]) -> DataFrame:
    """Adds the amounts of a secondary coverage to its primary coverage.

    For every pair, when one risk item of one transaction holds both coverage
    codes, the secondary amounts are added to the primary row and the secondary
    rows are removed. Rows without a coverage code are dropped, which matches
    the legacy behaviour.

    Args:
        df: Policy coverage rows.
        pairs: Pairs of [primary code, secondary code].

    Returns:
        The merged rows.
    """
    if not pairs:
        return df
    group = Window.partitionBy("RiskItemId", "InTransactionId")
    keyed = F.col("RiskItemId").isNotNull() & F.col("InTransactionId").isNotNull()
    helpers = ["_has_primary", "_has_secondary", "_secondary_onset", "_secondary_offset"]
    result = df.filter(F.col("CoverageCode").isNotNull())
    for primary, secondary in pairs:
        is_primary = F.col("CoverageCode") == F.lit(primary)
        is_secondary = F.col("CoverageCode") == F.lit(secondary)
        result = (
            result
            .withColumn("_has_primary", F.max(F.when(is_primary, 1).otherwise(0)).over(group))
            .withColumn("_has_secondary", F.max(F.when(is_secondary, 1).otherwise(0)).over(group))
            .withColumn(
                "_secondary_onset",
                F.sum(F.when(is_secondary, F.col("OnsetAmount")).otherwise(0)).over(group),
            )
            .withColumn(
                "_secondary_offset",
                F.sum(F.when(is_secondary, F.col("OffsetAmount")).otherwise(0)).over(group),
            )
        )
        merge_group = keyed & (F.col("_has_primary") == 1) & (F.col("_has_secondary") == 1)
        result = (
            result
            .filter(~(merge_group & is_secondary))
            .withColumn(
                "OnsetAmount",
                F.when(merge_group & is_primary, F.col("OnsetAmount") + F.col("_secondary_onset"))
                .otherwise(F.col("OnsetAmount")),
            )
            .withColumn(
                "OffsetAmount",
                F.when(merge_group & is_primary, F.col("OffsetAmount") + F.col("_secondary_offset"))
                .otherwise(F.col("OffsetAmount")),
            )
            .drop(*helpers)
        )
    return result


def apply_origami_prorata(ctx: Context, df: DataFrame) -> DataFrame:
    """Takes ProrataPercent from the question and answer table for Origami policies.

    Args:
        ctx: Pipeline context.
        df: Policy coverage rows.

    Returns:
        Rows with ProrataPercent replaced for Origami policies.
    """
    answers = (
        ctx.read_ods("question_answer")
        .filter(F.col("QuestionCode") == ctx.rules.prorata_question_code)
        .select(
            F.col("InTransactionId").alias("_qa_transaction"),
            F.col("TextValue").alias("_qa_prorata"),
        )
        .dropDuplicates(["_qa_transaction"])
    )
    return (
        df.join(answers, F.col("InTransactionId") == F.col("_qa_transaction"), "left")
        .withColumn(
            "ProrataPercent",
            F.when(F.col("PolicySource") == ctx.rules.origami_source, F.col("_qa_prorata"))
            .otherwise(F.col("ProrataPercent")),
        )
        .drop("_qa_transaction", "_qa_prorata")
    )


def build_policy_rows(ctx: Context) -> DataFrame:
    """Runs the whole policy extract stage.

    Args:
        ctx: Pipeline context.

    Returns:
        Cached policy coverage rows after exclusions, merging and prorata update.
    """
    rows = extract_policy_coverages(ctx)
    rows = exclude_policy_versions(ctx, rows)
    rows = exclude_duplicate_coverages(ctx, rows)
    rows = merge_paired_coverages(rows, ctx.rules.merge_coverage_pairs)
    rows = apply_origami_prorata(ctx, rows)
    rows = ctx.cache.keep(rows, "policy_rows")
    if ctx.cfg.enable_validations:
        logger.info("Policy coverage rows: %s", f"{rows.count():,}")
    return rows

## Stage 2: Onset and offset rows

In [ ]:
REFERENCE_COVERAGE_COLUMNS = [
    "CoverageId", "ParentCoverageId", "CoverageExtension", "CoverageGroup", "Limit1",
    "Limit2", "ASLOB", "Option1", "Exposure", "Deductible1", "ClaimCode", "CoverageInfoGUID",
]


def _offset_projection(
    risk_item_type: Column, class_code: Column, risk_item_id: Column, product_id: Column
) -> List[Column]:
    """Builds the output columns of reference rows for offset amounts.

    The reference row carries the coverage attributes of the transaction that the
    offset points to. The amounts stay with the current policy version.

    Args:
        risk_item_type: Expression for RiskItemType.
        class_code: Expression for ClassCode.
        risk_item_id: Expression for RiskItemId.
        product_id: Expression for ProductId.

    Returns:
        Columns in the order of the policy extract.
    """
    reference = {name: F.col(f"Ref_{name}").alias(name) for name in REFERENCE_COVERAGE_COLUMNS}
    return [
        F.col("PolicyNumber"),
        F.col("PolicyVersion"),
        F.col("Ref_InTransactionId").alias("InTransactionId"),
        F.col("ReferencePolicyVersion"),
        F.col("BookDate"),
        F.col("ProrataPercent"),
        F.col("PolicySource"),
        F.col("LOBCode"),
        risk_item_type.alias("RiskItemType"),
        F.col("UnitNumber"),
        class_code.alias("ClassCode"),
        F.col("PrimaryRiskState"),
        reference["CoverageId"],
        reference["ParentCoverageId"],
        risk_item_id.alias("RiskItemId"),
        product_id.alias("ProductId"),
        F.col("CoverageCode"),
        reference["CoverageExtension"],
        reference["CoverageGroup"],
        F.lit(0).cast("decimal(18,2)").alias("OnsetAmount"),
        F.col("OffsetAmount"),
        reference["Limit1"],
        reference["Limit2"],
        reference["ASLOB"],
        reference["Option1"],
        reference["Exposure"],
        reference["Deductible1"],
        reference["ClaimCode"],
        reference["CoverageInfoGUID"],
    ]


def build_offset_reference_rows(ctx: Context, policy_rows: DataFrame) -> DataFrame:
    """Creates offset rows that point to the coverage of the referenced transaction.

    An offset amount reverses an earlier premium. The row keeps the offset
    amount but takes its coverage attributes from the transaction identified by
    ReferencePolicyVersion, so that it is reported like the premium it reverses.

    Rows with a unit number and a risk item are matched on risk item. All other
    rows are matched on product, or on claim code for Origami policies.

    Args:
        ctx: Pipeline context.
        policy_rows: Policy coverage rows.

    Returns:
        Offset rows with the 29 policy columns.
    """
    rules = ctx.rules
    offsets = (
        policy_rows
        .filter(F.col("OffsetAmount").isNotNull() & (F.col("OffsetAmount") != 0))
        .withColumn("OnsetAmount", F.lit(0))
    )

    reference_header = ctx.read_ods("policy_detail").select(
        F.col("PolicyNumber").alias("_ref_policy_number"),
        F.col("PolicyVersion").alias("_ref_policy_version"),
        F.col("InTransactionId").alias("Ref_InTransactionId"),
    )
    with_reference = ctx.cache.keep(
        offsets.join(
            reference_header,
            (F.col("PolicyNumber") == F.col("_ref_policy_number"))
            & (F.col("_ref_policy_version") == F.col("ReferencePolicyVersion")),
            "left",
        ).drop("_ref_policy_number", "_ref_policy_version"),
        "offset_rows_with_reference",
    )
    reference_ids = (
        with_reference.select(F.col("Ref_InTransactionId").alias("InTransactionId"))
        .filter(F.col("InTransactionId").isNotNull())
    )

    has_risk_item = F.col("UnitNumber").isNotNull() & F.col("RiskItemId").isNotNull()
    risk_rows = with_reference.filter(has_risk_item)
    other_rows = with_reference.filter(~has_risk_item)

    # Risk item lookup of the referenced transaction.
    risk_lookup = (
        first_row_per_key(
            restrict_to_keys(ctx.read_ods("risk_item"), reference_ids, ["InTransactionId"]),
            ["InTransactionId", "UnitNumber", "RiskItemType"],
            ["RiskItemId"],
        )
        .select(
            F.col("InTransactionId").alias("_r_transaction"),
            F.col("UnitNumber").alias("_r_unit"),
            F.col("RiskItemType").alias("_r_type"),
            F.col("RiskItemId").alias("Ref_RiskItemId"),
            F.col("ClassCode").alias("Ref_ClassCode"),
            F.col("RiskItemType").alias("Ref_RiskItemType"),
        )
    )

    # Coverage lookups of the referenced transaction.
    coverage_source = restrict_to_keys(ctx.read_ods("coverage"), reference_ids, ["InTransactionId"])
    order = [F.col("ParentCoverageId").asc_nulls_first(), F.col("CoverageId").asc_nulls_first()]
    reference_columns = [F.col(name).alias(f"Ref_{name}") for name in REFERENCE_COVERAGE_COLUMNS]

    coverage_by_risk = first_row_per_key(
        coverage_source, ["InTransactionId", "RiskItemId", "CoverageCode"], order
    ).select(
        F.col("InTransactionId").alias("_c_transaction"),
        F.col("RiskItemId").alias("_c_risk_item"),
        F.col("CoverageCode").alias("_c_code"),
        *reference_columns,
    )
    coverage_by_product = first_row_per_key(
        coverage_source,
        ["InTransactionId", "ProductId", "CoverageCode", "CoverageExtension", "ClaimCode"],
        order,
    ).select(
        F.col("InTransactionId").alias("_p_transaction"),
        F.col("ProductId").alias("_p_product"),
        F.col("CoverageCode").alias("_p_code"),
        F.col("ClaimCode").alias("_p_claim_code"),
        *reference_columns,
    )

    risk_matched = (
        risk_rows.join(
            risk_lookup,
            (F.col("_r_transaction") == F.col("Ref_InTransactionId"))
            & (F.col("_r_unit") == F.col("UnitNumber"))
            & (F.col("_r_type") == F.col("RiskItemType")),
            "left",
        )
        .join(
            coverage_by_risk,
            (F.col("_c_transaction") == F.col("Ref_InTransactionId"))
            & (F.col("_c_risk_item") == F.col("Ref_RiskItemId"))
            & (F.col("_c_code") == F.col("CoverageCode")),
            "left",
        )
        .select(*_offset_projection(
            F.col("Ref_RiskItemType"), F.col("Ref_ClassCode"),
            F.col("Ref_RiskItemId"), F.lit(None),
        ))
    )

    product_projection = _offset_projection(
        F.col("RiskItemType"), F.lit(None), F.lit(None), F.col("ProductId")
    )
    not_origami = other_rows.filter(F.col("PolicySource") != rules.origami_source)
    origami = other_rows.filter(F.col("PolicySource") == rules.origami_source)
    product_matched = not_origami.join(
        coverage_by_product,
        (F.col("_p_transaction") == F.col("Ref_InTransactionId"))
        & (F.col("_p_product") == F.col("ProductId"))
        & (F.col("_p_code") == F.col("CoverageCode")),
        "left",
    ).select(*product_projection)
    claim_code_matched = origami.join(
        coverage_by_product,
        (F.col("_p_transaction") == F.col("Ref_InTransactionId"))
        & (F.col("_p_claim_code") == F.col("ClaimCode")),
        "left",
    ).select(*product_projection)

    return union_all([risk_matched, product_matched, claim_code_matched])


def apply_offset_parent_only_rules(df: DataFrame, conditions: Sequence[Dict[str, Any]]) -> DataFrame:
    """Keeps only child coverage rows for the listed policy and coverage combinations.

    For rows that match a condition set, rows without a ParentCoverageId are
    removed. Rows that do not match are kept. Rows where the match result is
    unknown (null) are removed, which matches the legacy behaviour.

    Args:
        df: Offset rows.
        conditions: Condition dictionaries, for example PolicyNumber, CoverageCode, ClaimCode.

    Returns:
        The filtered rows.
    """
    result = df
    for spec in conditions:
        matched = condition_from_dict(spec)
        keep = F.when(matched, F.col("ParentCoverageId").isNotNull()).when(~matched, F.lit(True))
        result = result.filter(keep)
    return result


def build_onset_and_offset_table(ctx: Context, policy_rows: DataFrame) -> DataFrame:
    """Builds and stores the table that holds onset rows and reference offset rows.

    Args:
        ctx: Pipeline context.
        policy_rows: Policy coverage rows.

    Returns:
        The stored table.
    """
    offsets = apply_offset_parent_only_rules(
        build_offset_reference_rows(ctx, policy_rows), ctx.rules.offset_parent_only_rules
    )
    onsets = (
        policy_rows
        .filter(F.col("OnsetAmount").isNotNull() & (F.col("OnsetAmount") != 0))
        .withColumn("OffsetAmount", F.lit(0))
    )
    return write_delta_table(ctx, onsets.unionByName(offsets), ctx.out(ctx.cfg.table_onset_offset))

## Stage 3: Claim valuation

In [ ]:
CLAIM_IDENTITY_COLUMNS = [
    "PolicyNumber", "TransactionNumber", "ClaimNumber", "ClaimantNumber", "ItemNumber",
    "ClaimCoverageName", "CoverageExtension", "LossDate", "AccidentYear",
    "DerivedClaimCoverageName",
]
CLAIM_COLUMNS = CLAIM_IDENTITY_COLUMNS + [
    "PaidCount", "ReserveCount", "PaidLossAmount", "ReserveAmount", "AllocatedLossAdjExp",
]
CLAIM_NULL_TYPES = {
    "PaidCount": "int", "ReserveCount": "int", "PaidLossAmount": "double",
    "ReserveAmount": "double", "AllocatedLossAdjExp": "double",
}
CORRECTION_KEYS = [
    "PolicyNumber", "TransactionNumber", "ClaimNumber", "ClaimantNumber",
    "ItemNumber", "DerivedClaimCoverageName",
]
MAPPING_COLUMNS = ["ClaimCoverageName", "CoverageExtension", "DerivedClaimCoverageName"]
MAPPING_VIEW = "niss_claim_coverage_mapping"
CORRECTION_MAPPING_VIEW = "niss_claim_coverage_mapping_correction"


def register_coverage_mapping_views(ctx: Context) -> None:
    """Registers the claim coverage mapping as two temporary views.

    The correction view is the normal mapping with Rules.correction_mapping_overrides
    applied. It is used only by the post year end payment correction.

    Args:
        ctx: Pipeline context.
    """
    rows = [tuple(row) for row in ctx.rules.claim_coverage_mapping]
    spark.createDataFrame(rows, MAPPING_COLUMNS).createOrReplaceTempView(MAPPING_VIEW)
    corrected = {(row[0], row[1]): row[2] for row in rows}
    for name, extension, derived in ctx.rules.correction_mapping_overrides:
        corrected[(name, extension)] = derived
    corrected_rows = [(key[0], key[1], value) for key, value in corrected.items()]
    spark.createDataFrame(corrected_rows, MAPPING_COLUMNS).createOrReplaceTempView(
        CORRECTION_MAPPING_VIEW
    )


def build_valuation_sql(ctx: Context, segment: ClaimSegment) -> str:
    """Builds the claim valuation query of one segment.

    Payments are summed for the reporting year, before the year and up to the
    end of the year. Incurred loss is summed before the year and up to the end of
    the valuation month. The reserve is read at the valuation month.

    Args:
        ctx: Pipeline context.
        segment: Claim segment.

    Returns:
        The SQL text.
    """
    cfg = ctx.cfg
    year_start = sql_literal(cfg.period_start.isoformat())
    year_end = sql_literal(cfg.period_end.isoformat())
    next_year = sql_literal(cfg.next_year_start.isoformat())
    valuation = sql_literal(segment.valuation_month.isoformat())
    incurred_end = sql_literal(segment.incurred_end_exclusive.isoformat())
    loss_year_filter = ""
    if segment.max_loss_year_exclusive is not None:
        loss_year_filter = f"AND YEAR(a.LossDate) < {int(segment.max_loss_year_exclusive)}"
    return f"""
SELECT /*+ BROADCAST(m) */
    a.PolicyNumber,
    a.TransactionNumber,
    a.ClaimNumber,
    a.ClaimantNumber,
    REPLACE(TRIM(a.ItemNumber), '.0', '') AS ItemNumber,
    a.ClaimCoverageName,
    a.CoverageExtension,
    a.LossDate,
    YEAR(a.LossDate) AS AccidentYear,
    m.DerivedClaimCoverageName,
    CAST(0 AS DECIMAL(2)) AS PaidCount,
    CAST(0 AS DECIMAL(2)) AS ReserveCount,
    SUM(CASE WHEN a.BeginningOfMonth BETWEEN {year_start} AND {year_end}
             THEN a.PaymentAmount ELSE 0 END) AS PaymentAmountDuration,
    SUM(CASE WHEN a.BeginningOfMonth < {year_start}
             THEN a.PaymentAmount ELSE 0 END) AS PaymentAmountAsOfPriorYearEnd,
    SUM(CASE WHEN a.BeginningOfMonth < {next_year}
             THEN a.PaymentAmount ELSE 0 END) AS PaymentAmountAsOfYearEnd,
    SUM(CASE WHEN a.BeginningOfMonth < {year_start}
             THEN a.LossIncurredNoIBNR ELSE 0 END) AS LossIncurredNoIBNRAsOfPriorYearEnd,
    SUM(CASE WHEN a.BeginningOfMonth < {incurred_end}
             THEN a.LossIncurredNoIBNR ELSE 0 END) AS LossIncurredNoIBNRAsOfValuation,
    SUM(CASE WHEN a.BeginningOfMonth = {valuation}
             THEN a.NetReserveAmount + a.ExcessDirectAmount ELSE 0 END) AS FinalReserveAmount
FROM {ctx.claims("claims_monthly")} a
LEFT JOIN {MAPPING_VIEW} m
    ON a.ClaimCoverageName = m.ClaimCoverageName
   AND a.CoverageExtension = m.CoverageExtension
WHERE a.CompanyCode = {sql_literal(cfg.company_code)}
  AND a.EASLobCode IN ({sql_list(segment.eas_lob_codes)})
  {loss_year_filter}
GROUP BY
    a.PolicyNumber, a.TransactionNumber, a.ClaimNumber, a.ClaimantNumber,
    a.ItemNumber, a.ClaimCoverageName, a.CoverageExtension,
    m.DerivedClaimCoverageName, a.LossDate, a.PropertyOrLiability
"""


def build_correction_sql(ctx: Context, segment: ClaimSegment) -> str:
    """Builds the query for payments made after year end up to the valuation month.

    Args:
        ctx: Pipeline context.
        segment: Claim segment with roll_back_to_year_end set.

    Returns:
        The SQL text.
    """
    cfg = ctx.cfg
    stub_start = sql_literal(cfg.next_year_start.isoformat())
    valuation = sql_literal(segment.valuation_month.isoformat())
    loss_year_filter = ""
    if segment.max_loss_year_exclusive is not None:
        loss_year_filter = f"AND YEAR(a.LossDate) < {int(segment.max_loss_year_exclusive)}"
    return f"""
SELECT /*+ BROADCAST(m) */
    a.EASLobCode,
    a.PolicyNumber,
    a.TransactionNumber,
    a.ClaimNumber,
    a.ClaimantNumber,
    REPLACE(TRIM(a.ItemNumber), '.0', '') AS ItemNumber,
    a.ClaimCoverageName,
    a.CoverageExtension,
    YEAR(a.LossDate) AS AccidentYear,
    m.DerivedClaimCoverageName,
    SUM(a.PaymentAmount + a.AllocatedLossAdjExp) AS PaidAmountAndAlaeAfterYearEnd
FROM {ctx.claims("claims_monthly")} a
LEFT JOIN {CORRECTION_MAPPING_VIEW} m
    ON a.ClaimCoverageName = m.ClaimCoverageName
   AND a.CoverageExtension = m.CoverageExtension
WHERE a.CompanyCode = {sql_literal(cfg.company_code)}
  AND a.EASLobCode IN ({sql_list(segment.eas_lob_codes)})
  {loss_year_filter}
  AND a.BeginningOfMonth BETWEEN {stub_start} AND {valuation}
GROUP BY
    a.EASLobCode, a.PolicyNumber, a.TransactionNumber, a.ClaimNumber,
    a.ClaimantNumber, a.ItemNumber, a.ClaimCoverageName, a.CoverageExtension,
    m.DerivedClaimCoverageName, a.LossDate, a.PropertyOrLiability
HAVING SUM(a.PaymentAmount + a.AllocatedLossAdjExp) != 0
"""


def build_allocated_loss_sql(ctx: Context) -> str:
    """Builds the allocated loss adjustment expense query for all segments.

    Args:
        ctx: Pipeline context.

    Returns:
        The SQL text.
    """
    cfg = ctx.cfg
    all_codes = [code for segment in cfg.segments for code in segment.eas_lob_codes]
    return f"""
SELECT /*+ BROADCAST(m) */
    a.PolicyNumber,
    a.TransactionNumber,
    a.ClaimNumber,
    a.ClaimantNumber,
    a.LossDate,
    YEAR(TO_DATE(a.LossDate, 'yyyyMMdd')) AS AccidentYear,
    a.PropertyOrLiability,
    a.ClaimCoverageName,
    a.CoverageExtension,
    a.LocationSequenceNumber,
    REPLACE(TRIM(a.ItemNumber), '.0', '') AS ItemNumber,
    SUM(a.AllocatedLossAdjExp) AS AllocatedLossAdjExp,
    m.DerivedClaimCoverageName
FROM {ctx.claims("claims_monthly")} a
LEFT JOIN {MAPPING_VIEW} m
    ON a.ClaimCoverageName = m.ClaimCoverageName
   AND a.CoverageExtension = m.CoverageExtension
WHERE a.EASLobCode IN ({sql_list(all_codes)})
  AND a.BeginningOfMonth BETWEEN {sql_literal(cfg.period_start.isoformat())}
                             AND {sql_literal(cfg.period_end.isoformat())}
  AND a.AllocatedLossAdjExp <> 0
GROUP BY
    a.PolicyNumber, a.TransactionNumber, a.ClaimNumber, a.ClaimantNumber,
    a.ClaimCoverageName, a.CoverageExtension, a.LocationSequenceNumber,
    REPLACE(TRIM(a.ItemNumber), '.0', ''), m.DerivedClaimCoverageName,
    a.LossDate, a.PropertyOrLiability
"""


def apply_post_year_end_correction(ctx: Context, df: DataFrame, segment: ClaimSegment) -> DataFrame:
    """Adds payments and expenses made after year end to the reserve of a segment.

    The reserve is read at the valuation month. When that month is after year
    end, payments made in between are added back so that the reserve reflects
    the position at the end of the reporting year.

    Args:
        ctx: Pipeline context.
        df: Valuation rows of the segment.
        segment: Claim segment.

    Returns:
        Rows with FinalReserveAmount adjusted. Row count is unchanged because the
        correction is aggregated to one row per key before the join.
    """
    if not segment.roll_back_to_year_end:
        return df
    if segment.valuation_month < ctx.cfg.next_year_start:
        logger.info("%s valuation month is inside the year, no roll back needed", segment.name)
        return df
    correction = (
        spark.sql(build_correction_sql(ctx, segment))
        .groupBy(*CORRECTION_KEYS)
        .agg(
            F.sum(F.coalesce(F.col("PaidAmountAndAlaeAfterYearEnd"), F.lit(0)))
            .alias("PaidAmountAndAlaeAfterYearEnd")
        )
    )
    return (
        df.join(correction, on=CORRECTION_KEYS, how="left")
        .withColumn(
            "FinalReserveAmount",
            F.coalesce(F.col("FinalReserveAmount"), F.lit(0))
            + F.coalesce(F.col("PaidAmountAndAlaeAfterYearEnd"), F.lit(0)),
        )
        .drop("PaidAmountAndAlaeAfterYearEnd")
    )


def project_claim_columns(df: DataFrame, provided: Sequence[str]) -> DataFrame:
    """Selects the common claim columns and fills the missing ones with typed nulls.

    Args:
        df: Source DataFrame.
        provided: Claim columns that exist in the source DataFrame.

    Returns:
        DataFrame with exactly CLAIM_COLUMNS.
    """
    columns = []
    for name in CLAIM_COLUMNS:
        if name in provided:
            columns.append(F.col(name))
        else:
            columns.append(F.lit(None).cast(CLAIM_NULL_TYPES[name]).alias(name))
    return df.select(*columns)


def split_paid_and_reserve_rows(ctx: Context, valuation: DataFrame) -> Tuple[DataFrame, DataFrame]:
    """Derives paid loss rows and reserve rows with their claim counts.

    Args:
        ctx: Pipeline context.
        valuation: Valuation rows of all segments.

    Returns:
        Tuple of (paid loss rows, reserve rows) in the common claim layout.
    """
    duration = F.col("PaymentAmountDuration")
    as_of_year_end = F.col("PaymentAmountAsOfYearEnd")
    as_of_prior = F.col("PaymentAmountAsOfPriorYearEnd")
    incurred_valuation = F.col("LossIncurredNoIBNRAsOfValuation")
    incurred_prior = F.col("LossIncurredNoIBNRAsOfPriorYearEnd")

    paid = (
        valuation
        .filter(duration != 0)
        .withColumn("PaidLossAmount", duration)
        .withColumn(
            "PaidCount",
            F.when((duration == as_of_year_end) & (duration > 0) & (as_of_year_end > 0), 1)
            .when((duration < 0) & (as_of_prior > 0) & ((duration + as_of_prior) == 0), -1)
            .otherwise(0),
        )
    )
    reserve = (
        valuation
        .filter(
            ~(
                (as_of_year_end == 0)
                & (incurred_valuation == 0)
                & (F.year("LossDate") == ctx.cfg.calendar_year)
            )
        )
        .withColumn("ReserveAmount", F.col("FinalReserveAmount"))
        .filter(F.col("ReserveAmount") != 0)
        .withColumn(
            "ReserveCount",
            F.when((as_of_year_end == 0) & (incurred_valuation != 0), 1)
            .when(
                (as_of_year_end == 0) & (incurred_valuation == 0)
                & (as_of_prior == 0) & (incurred_prior != 0),
                -1,
            )
            .otherwise(0),
        )
    )
    paid_columns = CLAIM_IDENTITY_COLUMNS + ["PaidCount", "ReserveCount", "PaidLossAmount"]
    reserve_columns = CLAIM_IDENTITY_COLUMNS + ["PaidCount", "ReserveCount", "ReserveAmount"]
    return (
        project_claim_columns(paid, paid_columns),
        project_claim_columns(reserve, reserve_columns),
    )


def build_claim_rows(ctx: Context) -> DataFrame:
    """Builds paid loss, reserve and allocated expense rows for the reporting year.

    Args:
        ctx: Pipeline context.

    Returns:
        Claim rows in the common layout, with TransactionNumber as integer.
    """
    register_coverage_mapping_views(ctx)
    segments = []
    for segment in ctx.cfg.segments:
        valuation = spark.sql(build_valuation_sql(ctx, segment))
        segments.append(apply_post_year_end_correction(ctx, valuation, segment))
    valuation_all = union_all(segments).withColumn(
        "TransactionNumber", F.col("TransactionNumber").cast("int")
    )
    paid, reserve = split_paid_and_reserve_rows(ctx, valuation_all)

    allocated = spark.sql(build_allocated_loss_sql(ctx)).withColumn(
        "TransactionNumber", F.col("TransactionNumber").cast("int")
    )
    allocated = project_claim_columns(
        allocated, CLAIM_IDENTITY_COLUMNS + ["AllocatedLossAdjExp"]
    )
    return union_all([paid, reserve, allocated])


def normalize_claim_coverage(ctx: Context, claims: DataFrame) -> DataFrame:
    """Aligns coverage extension and derived coverage name with policy conventions.

    Args:
        ctx: Pipeline context.
        claims: Claim rows.

    Returns:
        Claim rows with updated CoverageExtension and DerivedClaimCoverageName.
    """
    rules = ctx.rules
    prefix = rules.origami_policy_prefix
    not_origami = ~F.col("PolicyNumber").startswith(prefix)

    ba_plus = rules.claim_ba_plus_rule
    collapse = rules.claim_extension_collapse
    extension = (
        F.when(
            (F.col("ClaimCoverageName") == ba_plus["claim_coverage_name"]) & not_origami,
            ba_plus["extension"],
        ).otherwise(F.col("CoverageExtension"))
    )
    claims = claims.withColumn("CoverageExtension", extension)
    claims = claims.withColumn(
        "CoverageExtension",
        F.when(F.col("CoverageExtension").isin(*collapse["extensions"]), collapse["to"])
        .otherwise(F.col("CoverageExtension")),
    )
    med = rules.claim_med_exp_rule
    return claims.withColumn(
        "DerivedClaimCoverageName",
        F.when(
            (F.col("DerivedClaimCoverageName") == med["derived_name"])
            & (F.col("CoverageExtension") == med["extension"])
            & not_origami,
            med["new_name"],
        ).otherwise(F.col("DerivedClaimCoverageName")),
    )


def filter_claims_by_policy_term(ctx: Context, claims: DataFrame) -> DataFrame:
    """Keeps claims whose policy term starts on or after the configured date.

    Args:
        ctx: Pipeline context.
        claims: Claim rows.

    Returns:
        Claim rows with the same columns.
    """
    policy = (
        ctx.read_ods("policy_detail")
        .filter(F.col("TermEffectiveDate") >= F.lit(ctx.cfg.claim_term_effective_from))
        .select(
            F.col("PolicyNumber").alias("_term_policy_number"),
            F.col("PolicyVersion").cast("int").alias("_term_policy_version"),
        )
    )
    return (
        claims.join(
            policy,
            (F.col("PolicyNumber") == F.col("_term_policy_number"))
            & (F.col("TransactionNumber").cast("int") == F.col("_term_policy_version")),
            "inner",
        )
        .select(*claims.columns)
    )


def attach_claim_guids(ctx: Context, claims: DataFrame) -> DataFrame:
    """Adds coverage and risk item identifiers from the claims fact table.

    When the fact table has no transaction number, the number is taken from the
    monthly claims table for the same policy, claim and claimant.

    Args:
        ctx: Pipeline context.
        claims: Cached claim rows.

    Returns:
        Claim rows with ParentCoverageGUID, ChildCoverageGUID, CauseOfLoss and
        ParentRiskitemGUID.
    """
    cfg = ctx.cfg
    keys = ["PolicyNumber", "ClaimNumber", "ClaimantNumber"]
    guid_source = (
        ctx.read_claims("claims_fact")
        .filter((F.col("CompanyCode") == cfg.company_code) & (F.col("LOBCode") == cfg.lob_code))
        .select(
            "PolicyNumber", "TransactionNumber", "ClaimNumber", "ClaimantNumber", "ItemNumber",
            "ClaimCoverageName", "CoverageExtension", "CauseOfLoss", "ParentCoverageGUID",
            "ChildCoverageGUID", "ParentRiskitemGUID",
        )
    )
    guid_source = restrict_to_keys(guid_source, claims, ["PolicyNumber"])

    missing_transaction = (
        guid_source.filter(F.col("TransactionNumber").isNull()).select(*keys).distinct()
    )
    fallback = (
        restrict_to_keys(
            ctx.read_claims("claims_monthly").select(
                *keys, F.col("TransactionNumber").alias("_fallback_transaction")
            ),
            missing_transaction, keys, broadcast=False,
        )
        .dropDuplicates(keys)
    )
    guid_rows = (
        guid_source.join(fallback, keys, "left")
        .withColumn(
            "TransactionNumber",
            F.coalesce(F.col("TransactionNumber"), F.col("_fallback_transaction")),
        )
        .drop("_fallback_transaction")
        .dropDuplicates([
            "PolicyNumber", "TransactionNumber", "ClaimNumber", "ClaimantNumber",
            "ItemNumber", "ClaimCoverageName", "CoverageExtension", "ParentRiskitemGUID",
        ])
    )
    mapping_columns = [
        "PolicyNumber", "TransactionNumber", "ClaimNumber", "ClaimantNumber", "ItemNumber",
        "ClaimCoverageName", "CoverageExtension", "ParentCoverageGUID", "ChildCoverageGUID",
        "CauseOfLoss", "ParentRiskitemGUID",
    ]
    mapping = guid_rows.select(*[F.col(name).alias(f"m_{name}") for name in mapping_columns])
    condition = (
        (F.col("PolicyNumber") == F.col("m_PolicyNumber"))
        & (F.col("ClaimNumber") == F.col("m_ClaimNumber"))
        & (F.col("ClaimantNumber") == F.col("m_ClaimantNumber"))
        & (F.col("ItemNumber").cast("int") == F.col("m_ItemNumber").cast("int"))
        & (F.col("DerivedClaimCoverageName") == F.col("m_ClaimCoverageName"))
        & (F.col("CoverageExtension") == F.col("m_CoverageExtension"))
        & (F.col("TransactionNumber").cast("int") == F.col("m_TransactionNumber").cast("int"))
    )
    return claims.join(mapping, condition, "left").select(
        *claims.columns,
        F.col("m_ParentCoverageGUID").alias("ParentCoverageGUID"),
        F.col("m_ChildCoverageGUID").alias("ChildCoverageGUID"),
        F.col("m_CauseOfLoss").alias("CauseOfLoss"),
        F.col("m_ParentRiskitemGUID").alias("ParentRiskitemGUID"),
    )


def prepare_claims(ctx: Context) -> DataFrame:
    """Runs the claim stage up to the cached, policy filtered claim rows.

    Args:
        ctx: Pipeline context.

    Returns:
        Cached claim rows with identifiers, ready for policy matching.
    """
    claims = build_claim_rows(ctx)
    claims = normalize_claim_coverage(ctx, claims)
    claims = filter_claims_by_policy_term(ctx, claims)
    if ctx.cfg.drop_claim_rows_with_null_reserve:
        # Legacy behaviour: a null ReserveAmount fails the comparison, so paid loss
        # and expense rows are removed together with negative reserves.
        claims = claims.filter(F.col("ReserveAmount") >= 0)
    else:
        claims = claims.filter(F.col("ReserveAmount").isNull() | (F.col("ReserveAmount") >= 0))
    claims = ctx.cache.keep(claims, "claims_base")
    if ctx.cfg.enable_validations:
        logger.info("Claim rows after policy term and reserve filters: %s", f"{claims.count():,}")
    return ctx.cache.keep(attach_claim_guids(ctx, claims), "claims_with_guids")

## Stage 4: Claim to policy mapping

In [ ]:
POLICY_HEADER_COLUMNS = [
    "PolicyVersion", "InTransactionId", "ReferencePolicyVersion", "PolicySource",
    "BookDate", "PrimaryRiskState", "ProrataPercent",
]
CLAIM_POLICY_KEYS = [
    "PolicyNumber", "TransactionNumber", "ClaimNumber", "ClaimantNumber",
    "ClaimCoverageName", "CoverageExtension",
]
CLAIM_GUID_COLUMNS = ["ParentCoverageGUID", "ChildCoverageGUID", "CauseOfLoss", "ParentRiskitemGUID"]
FALLBACK_COLUMNS = (
    CLAIM_COLUMNS + CLAIM_GUID_COLUMNS
    + [
        "PolicyVersion", "InTransactionId", "ReferencePolicyVersion", "PolicySource",
        "BookDate", "ProrataPercent", "PrimaryRiskState", "LOBCode", "RiskItemType",
        "UnitNumber", "ClassCode", "CoverageId", "ParentCoverageId", "RiskItemId",
        "ProductId", "CoverageCode", "PolicyCoverageExtension", "CoverageGroup",
        "OnsetAmount", "OffsetAmount", "Limit1", "Limit2", "ASLOB", "Option1",
        "Exposure", "Deductible1", "ClaimCode", "PolicyCoverageInfoGUID",
    ]
)
# Policy amounts and proration are not carried for claim rows. The untyped nulls
# take the type of the matching column when the DataFrames are combined.
FALLBACK_NULL_COLUMNS = {
    "ProrataPercent": None, "Exposure": None, "OnsetAmount": "double", "OffsetAmount": "double",
}


def _fallback_projection() -> List[Column]:
    """Builds the output column list of the fallback matching flows."""
    columns = []
    for name in FALLBACK_COLUMNS:
        if name in FALLBACK_NULL_COLUMNS:
            null_type = FALLBACK_NULL_COLUMNS[name]
            value = F.lit(None) if null_type is None else F.lit(None).cast(null_type)
            columns.append(value.alias(name))
        else:
            columns.append(F.col(name))
    return columns


def _policy_coverage_columns() -> List[Column]:
    """Coverage columns used when claim rows are matched to policy coverages."""
    return [
        F.col("CoverageId"), F.col("ParentCoverageId"), F.col("RiskItemId"), F.col("ProductId"),
        F.col("CoverageCode"), F.col("CoverageExtension").alias("PolicyCoverageExtension"),
        F.col("CoverageGroup"), F.col("OnsetAmount"), F.col("OffsetAmount"), F.col("Limit1"),
        F.col("Limit2"), F.col("ASLOB"), F.col("Option1"), F.col("Exposure"),
        F.col("Deductible1"), F.col("ClaimCode"),
        F.col("CoverageInfoGUID").alias("PolicyCoverageInfoGUID"),
    ]


def enrich_claims_with_policy(ctx: Context, claims: DataFrame) -> DataFrame:
    """Joins claim rows to the policy coverage that their identifiers point to.

    A claim carries the identifiers of the policy coverage and risk item that it
    was reported against. Those identifiers are looked up inside the policy
    transaction that matches the claim policy number and transaction number.

    Args:
        ctx: Pipeline context.
        claims: Claim rows with identifiers.

    Returns:
        Claim rows with policy header, risk item and coverage columns.
    """
    header = (
        restrict_to_keys(ctx.read_ods("policy_detail"), claims, ["PolicyNumber"])
        .select(F.col("PolicyNumber").alias("_h_policy_number"), *POLICY_HEADER_COLUMNS)
    )
    with_header = ctx.cache.keep(
        claims.join(
            header,
            (F.col("PolicyNumber") == F.col("_h_policy_number"))
            & (F.col("TransactionNumber").cast("int") == F.col("PolicyVersion").cast("int")),
            "left",
        ).drop("_h_policy_number"),
        "claims_with_policy_header",
    )
    transaction_ids = with_header.select("InTransactionId").filter(F.col("InTransactionId").isNotNull())

    product = restrict_to_keys(ctx.read_ods("product"), transaction_ids, ["InTransactionId"]).select(
        F.col("InTransactionId").alias("_pi_transaction"), "LOBCode"
    )
    if ctx.cfg.drop_duplicate_rows:
        # Identical rows are removed later, so repeated product rows add nothing.
        product = product.distinct()
    risk_item = restrict_to_keys(ctx.read_ods("risk_item"), transaction_ids, ["InTransactionId"]).select(
        F.col("InTransactionId").alias("_ri_transaction"),
        F.upper(F.col("RiskItemInfoGUID")).alias("_ri_guid"),
        "RiskItemType", "UnitNumber", "ClassCode",
    )
    coverage = restrict_to_keys(ctx.read_ods("coverage"), transaction_ids, ["InTransactionId"]).select(
        F.col("InTransactionId").alias("_ci_transaction"),
        F.upper(F.col("CoverageInfoGUID")).alias("_ci_guid"),
        *_policy_coverage_columns(),
    )
    return (
        with_header
        .join(product, F.col("InTransactionId") == F.col("_pi_transaction"), "left")
        .join(
            risk_item,
            (F.col("InTransactionId") == F.col("_ri_transaction"))
            & (F.upper(F.col("ParentRiskitemGUID")) == F.col("_ri_guid")),
            "left",
        )
        .join(
            coverage,
            (F.col("InTransactionId") == F.col("_ci_transaction"))
            & (F.upper(F.col("ParentCoverageGUID")) == F.col("_ci_guid")),
            "left",
        )
        .drop("_pi_transaction", "_ri_transaction", "_ri_guid", "_ci_transaction", "_ci_guid")
    )


def _existing_guids(
    policy_table: DataFrame, policy_column: str, claims: DataFrame, claim_column: str, flag: str
) -> DataFrame:
    """Finds which claim identifiers exist in a policy table.

    Args:
        policy_table: Policy table that holds the identifiers.
        policy_column: Identifier column in the policy table.
        claims: Claims with normalized identifiers.
        claim_column: Identifier column in the claims.
        flag: Name of the column that marks identifiers that were found.

    Returns:
        A small DataFrame of (claim_column, flag) for identifiers that exist.
    """
    wanted = claims.select(claim_column).filter(F.col(claim_column).isNotNull()).distinct()
    return (
        policy_table.select(F.upper(F.trim(F.col(policy_column))).alias(claim_column))
        .join(F.broadcast(wanted), claim_column, "left_semi")
        .distinct()
        .withColumn(flag, F.lit(True))
    )


def find_unmatched_claims(ctx: Context, claims: DataFrame) -> DataFrame:
    """Finds claims whose coverage or risk item identifier cannot be used.

    A claim is unmatched when an identifier is missing on the claim or when it
    does not exist in the policy tables.

    Args:
        ctx: Pipeline context.
        claims: Claim rows with identifiers.

    Returns:
        Unmatched claim rows with trimmed, upper case identifiers.
    """
    normalized = (
        claims
        .withColumn("ParentCoverageGUID", F.upper(F.trim(F.col("ParentCoverageGUID"))))
        .withColumn("ParentRiskitemGUID", F.upper(F.trim(F.col("ParentRiskitemGUID"))))
    )
    coverage_found = _existing_guids(
        ctx.read_ods("coverage"), "CoverageInfoGUID", normalized,
        "ParentCoverageGUID", "_coverage_guid_found",
    )
    risk_found = _existing_guids(
        ctx.read_ods("risk_item"), "RiskItemInfoGUID", normalized,
        "ParentRiskitemGUID", "_risk_guid_found",
    )
    checked = (
        normalized
        .join(F.broadcast(coverage_found), "ParentCoverageGUID", "left")
        .join(F.broadcast(risk_found), "ParentRiskitemGUID", "left")
    )
    unusable = (
        F.col("ParentCoverageGUID").isNull() | F.col("ParentRiskitemGUID").isNull()
        | F.col("_coverage_guid_found").isNull() | F.col("_risk_guid_found").isNull()
    )
    return checked.filter(unusable).select(*claims.columns)


def _fallback_by_unit_number(ctx: Context, claims: DataFrame) -> DataFrame:
    """Matches claims with an item number to the policy coverage of the same unit.

    The match uses policy number, policy version, printed unit number, coverage
    code and coverage extension.

    Args:
        ctx: Pipeline context.
        claims: Unmatched claims whose item number is not zero.

    Returns:
        Claim rows in the fallback column layout.
    """
    claim_keys = claims.select(
        F.col("PolicyNumber").alias("_k_policy_number"),
        F.col("TransactionNumber").cast("int").alias("_k_version"),
    )
    header = (
        ctx.read_ods("policy_detail")
        .join(
            F.broadcast(claim_keys.distinct()),
            (F.col("PolicyNumber") == F.col("_k_policy_number"))
            & (F.col("PolicyVersion").cast("int") == F.col("_k_version")),
            "left_semi",
        )
        .select(F.col("PolicyNumber").alias("_pol_policy_number"), *POLICY_HEADER_COLUMNS)
    )
    header_ids = header.select("InTransactionId")
    risk_item = restrict_to_keys(ctx.read_ods("risk_item"), header_ids, ["InTransactionId"]).select(
        "InTransactionId", "RiskItemId", "RiskItemType", "UnitNumber", "ClassCode"
    )
    coverage = restrict_to_keys(ctx.read_ods("coverage"), header_ids, ["InTransactionId"]).select(
        "InTransactionId", *_policy_coverage_columns()
    )
    unit_numbers = (
        restrict_to_keys(ctx.read_ods("question_answer"), header_ids, ["InTransactionId"])
        .filter(F.col("QuestionCode") == ctx.rules.unit_number_question_code)
        .select("InTransactionId", "RiskItemId", F.col("TextValue").alias("_pol_item_number"))
    )
    policy_items = (
        header.join(risk_item, "InTransactionId")
        .join(coverage, ["InTransactionId", "RiskItemId"])
        .join(unit_numbers, ["InTransactionId", "RiskItemId"])
    )
    policy_items = first_row_per_key(
        policy_items,
        ["_pol_policy_number", "PolicyVersion", "_pol_item_number", "CoverageCode",
         "PolicyCoverageExtension"],
        [F.col("CoverageId").asc_nulls_first()],
    )
    matched = claims.join(
        policy_items,
        (F.col("PolicyNumber") == F.col("_pol_policy_number"))
        & (F.col("TransactionNumber").cast("int") == F.col("PolicyVersion").cast("int"))
        & (F.col("ItemNumber").cast("int") == F.col("_pol_item_number").cast("int"))
        & (F.col("DerivedClaimCoverageName") == F.col("CoverageCode"))
        & (F.col("CoverageExtension") == F.col("PolicyCoverageExtension")),
        "left",
    )
    return matched.withColumn("LOBCode", F.lit(ctx.cfg.lob_code)).select(*_fallback_projection())


def _fallback_by_claim_coverage(ctx: Context, claims: DataFrame) -> DataFrame:
    """Matches claims without an item number to the policy coverage by name.

    The match uses the policy transaction and the claim coverage name against
    the policy coverage code. No risk item is assigned.

    Args:
        ctx: Pipeline context.
        claims: Unmatched claims whose item number is zero.

    Returns:
        Claim rows in the fallback column layout.
    """
    claim_keys = claims.select(
        F.col("PolicyNumber").alias("_k_policy_number"),
        F.col("TransactionNumber").alias("_k_version"),
    )
    header = (
        ctx.read_ods("policy_detail")
        .join(
            F.broadcast(claim_keys.distinct()),
            (F.col("PolicyNumber") == F.col("_k_policy_number"))
            & (F.col("PolicyVersion") == F.col("_k_version")),
            "left_semi",
        )
        .select(F.col("PolicyNumber").alias("_pol_policy_number"), *POLICY_HEADER_COLUMNS)
    )
    coverage = (
        restrict_to_keys(ctx.read_ods("coverage"), header.select("InTransactionId"), ["InTransactionId"])
        .select(F.col("InTransactionId").alias("_c_transaction"), *_policy_coverage_columns())
    )
    with_header = claims.join(
        header,
        (F.col("PolicyNumber") == F.col("_pol_policy_number"))
        & (F.col("TransactionNumber") == F.col("PolicyVersion")),
        "left",
    )
    matched = with_header.join(
        coverage,
        (F.col("InTransactionId") == F.col("_c_transaction"))
        & (F.col("ClaimCoverageName") == F.col("CoverageCode")),
        "left",
    )
    return (
        matched
        .withColumn("LOBCode", F.lit(ctx.cfg.lob_code))
        .withColumn("RiskItemType", F.lit(None))
        .withColumn("UnitNumber", F.lit(None))
        .withColumn("ClassCode", F.lit(None))
        .select(*_fallback_projection())
    )


def build_unit_fallback_rows(ctx: Context, unmatched: DataFrame) -> DataFrame:
    """Matches unmatched claims to policy coverages with the fallback rules.

    Args:
        ctx: Pipeline context.
        unmatched: Unmatched claim rows.

    Returns:
        Fallback rows that replace the claim rows with the same keys.
    """
    with_item = unmatched.filter(F.col("ItemNumber") != 0)
    without_item = unmatched.filter(F.col("ItemNumber") == 0)
    return union_all([
        _fallback_by_unit_number(ctx, with_item),
        _fallback_by_claim_coverage(ctx, without_item),
    ])


def merge_claim_policy_rows(enriched: DataFrame, fallback: DataFrame) -> DataFrame:
    """Replaces enriched claim rows by their fallback rows and renames columns.

    Args:
        enriched: Claim rows joined through identifiers.
        fallback: Claim rows matched with the fallback rules.

    Returns:
        Combined rows. CoverageExtension holds the policy extension and
        ClaimCoverageExtension holds the claim extension.
    """
    kept = enriched.join(fallback.select(*CLAIM_POLICY_KEYS).distinct(), CLAIM_POLICY_KEYS, "left_anti")
    return (
        kept.unionByName(fallback)
        .withColumnRenamed("CoverageExtension", "ClaimCoverageExtension")
        .withColumnRenamed("PolicyCoverageInfoGUID", "CoverageInfoGUID")
        .withColumnRenamed("PolicyCoverageExtension", "CoverageExtension")
    )


def build_claim_policy_table(ctx: Context, claims: DataFrame) -> DataFrame:
    """Maps claim rows to policy coverages and stores the result.

    Args:
        ctx: Pipeline context.
        claims: Cached claim rows with identifiers.

    Returns:
        The stored table.
    """
    enriched = enrich_claims_with_policy(ctx, claims)
    unmatched = ctx.cache.keep(find_unmatched_claims(ctx, claims), "unmatched_claims")
    unmatched = apply_value_rules(
        unmatched, "DerivedClaimCoverageName", ctx.rules.derived_coverage_overrides
    )
    fallback = build_unit_fallback_rows(ctx, unmatched)
    merged = merge_claim_policy_rows(enriched, fallback)
    return write_delta_table(ctx, merged, ctx.out(ctx.cfg.table_claim_policy))

## Stage 5: Zip codes

In [ ]:
def log_column_differences(left: DataFrame, right: DataFrame, left_name: str, right_name: str) -> None:
    """Logs the columns that exist in only one of two DataFrames.

    Args:
        left: First DataFrame.
        right: Second DataFrame.
        left_name: Label of the first DataFrame.
        right_name: Label of the second DataFrame.
    """
    only_left = sorted(set(left.columns) - set(right.columns))
    only_right = sorted(set(right.columns) - set(left.columns))
    logger.info("Columns only in %s: %s", left_name, only_left)
    logger.info("Columns only in %s: %s", right_name, only_right)


def combine_policy_and_claim_rows(ctx: Context) -> DataFrame:
    """Combines premium rows and claim rows into one DataFrame.

    Claim rows carry no premium, so their amounts and proration are set to zero
    or null. Columns that exist on one side only are filled with nulls.

    Args:
        ctx: Pipeline context.

    Returns:
        Combined rows.
    """
    premium_rows = spark.table(ctx.out(ctx.cfg.table_onset_offset))
    claim_rows = spark.table(ctx.out(ctx.cfg.table_claim_policy)).withColumns({
        "OffsetAmount": F.lit(0),
        "OnsetAmount": F.lit(0),
        "Exposure": F.lit(None),
        "ProrataPercent": F.lit(None),
    })
    if ctx.cfg.enable_validations:
        log_column_differences(premium_rows, claim_rows, "premium rows", "claim rows")
    return premium_rows.unionByName(claim_rows, allowMissingColumns=True)


def _address_rows(ctx: Context, transactions: DataFrame) -> DataFrame:
    """Reads policy addresses with their zip code for a set of transactions.

    Args:
        ctx: Pipeline context.
        transactions: DataFrame with InTransactionId.

    Returns:
        Rows with InTransactionId, RiskItemId, AddressUse, ZipCode and ZipExt.
    """
    policy_address = restrict_to_keys(
        ctx.read_ods("policy_address"), transactions, ["InTransactionId"]
    ).select(
        "InTransactionId", "RiskItemId", "AddressUse", F.col("AddressID").alias("_address_id")
    )
    party_address = ctx.read_ods("party_address").select(
        F.col("AddressId").alias("_party_address_id"), "ZipCode", "ZipExt"
    )
    return policy_address.join(
        party_address, F.col("_address_id") == F.col("_party_address_id")
    ).select("InTransactionId", "RiskItemId", "AddressUse", "ZipCode", "ZipExt")


def _vehicle_items(ctx: Context, transactions: DataFrame) -> DataFrame:
    """Reads the vehicle risk items of a set of transactions."""
    return (
        restrict_to_keys(ctx.read_ods("risk_item"), transactions, ["InTransactionId"])
        .filter(F.col("RiskItemType") == ctx.rules.risk_type_vehicle)
        .select("InTransactionId", "RiskItemId")
    )


def _first_zip_per_transaction(rows: DataFrame) -> DataFrame:
    """Reduces address rows to the first available zip code per transaction."""
    return rows.groupBy("InTransactionId").agg(
        F.first("ZipCode", ignorenulls=True).alias("ZipCode"),
        F.first("ZipExt", ignorenulls=True).alias("ZipExt"),
    )


def build_ibmi_zip_lookup(ctx: Context, data: DataFrame) -> DataFrame:
    """Finds the zip code of IBMi transactions from the primary location.

    The vehicles whose location number equals the primary location value supply
    the address.

    Args:
        ctx: Pipeline context.
        data: Combined rows.

    Returns:
        One row per transaction with ZipCode and ZipExt.
    """
    rules = ctx.rules
    transactions = (
        data.filter(F.col("PolicySource") == rules.ibmi_source).select("InTransactionId").distinct()
    )
    primary_location = (
        restrict_to_keys(ctx.read_ods("question_answer"), transactions, ["InTransactionId"])
        .filter(
            (F.col("QuestionCode") == rules.location_question_code)
            & (F.col("TextValue") == rules.primary_location_value)
        )
        .select("InTransactionId", "RiskItemId")
    )
    rows = (
        _vehicle_items(ctx, transactions)
        .join(primary_location, ["InTransactionId", "RiskItemId"], "inner")
        .join(_address_rows(ctx, transactions), ["InTransactionId", "RiskItemId"], "inner")
    )
    return _first_zip_per_transaction(rows)


def build_origami_zip_lookup(ctx: Context, data: DataFrame) -> DataFrame:
    """Finds the zip code of Origami transactions from risk and garage addresses.

    Args:
        ctx: Pipeline context.
        data: Combined rows.

    Returns:
        One row per transaction with ZipCode and ZipExt.
    """
    rules = ctx.rules
    transactions = (
        data.filter(F.col("PolicySource") == rules.origami_source).select("InTransactionId").distinct()
    )
    addresses = _address_rows(ctx, transactions).filter(
        F.col("AddressUse").isin(*rules.origami_address_uses)
    )
    rows = (
        addresses.join(_vehicle_items(ctx, transactions), ["InTransactionId", "RiskItemId"], "inner")
        .filter(F.col("ZipCode").isNotNull() & (F.trim(F.col("ZipCode")) != ""))
    )
    return _first_zip_per_transaction(rows)


def attach_zip_codes(ctx: Context, data: DataFrame) -> DataFrame:
    """Adds ZipCode and ZipExt to every row.

    The IBMi lookup is used first. The Origami lookup fills values that are
    still missing.

    Args:
        ctx: Pipeline context.
        data: Combined rows.

    Returns:
        Rows with ZipCode and ZipExt appended.
    """
    ibmi = build_ibmi_zip_lookup(ctx, data).select(
        "InTransactionId",
        F.col("ZipCode").alias("_ibmi_zip"),
        F.col("ZipExt").alias("_ibmi_ext"),
    )
    origami = build_origami_zip_lookup(ctx, data).select(
        "InTransactionId",
        F.col("ZipCode").alias("_origami_zip"),
        F.col("ZipExt").alias("_origami_ext"),
    )
    joined = data.join(ibmi, "InTransactionId", "left").join(origami, "InTransactionId", "left")
    return joined.select(
        *data.columns,
        F.coalesce(F.col("_ibmi_zip"), F.col("_origami_zip")).alias("ZipCode"),
        F.coalesce(F.col("_ibmi_ext"), F.col("_origami_ext")).alias("ZipExt"),
    )


def build_address_table(ctx: Context) -> DataFrame:
    """Combines premium and claim rows, adds zip codes and stores the result.

    Args:
        ctx: Pipeline context.

    Returns:
        The stored table.
    """
    combined = combine_policy_and_claim_rows(ctx)
    return write_delta_table(
        ctx, attach_zip_codes(ctx, combined), ctx.out(ctx.cfg.table_onset_offset_address)
    )

## Stage 6: Transformations

In [ ]:
def apply_coverage_attribute_rules(ctx: Context, df: DataFrame) -> DataFrame:
    """Applies coverage extension, coverage group and ASLOB rules by policy source.

    Args:
        ctx: Pipeline context.
        df: Rows with policy and claim data.

    Returns:
        Rows with CoverageExtension, CoverageGroup and ASLOB updated.
    """
    rules = ctx.rules
    ibmi = {"PolicySource": rules.ibmi_source}
    origami = {"PolicySource": rules.origami_source}

    extension_rules = mapping_rules(ibmi, "CoverageCode", rules.ibmi_extension_by_coverage)
    extension_rules.append({
        "when": {**ibmi, "CoverageCode": rules.ibmi_cbi_to_bi_coverages, "CoverageExtension": "CBI"},
        "value": "BI",
    })
    extension_rules += mapping_rules(origami, "CoverageCode", rules.origami_extension_by_coverage)

    result = apply_value_rules(df, "CoverageExtension", extension_rules)
    result = apply_value_rules(
        result, "CoverageGroup", mapping_rules(ibmi, "CoverageCode", rules.ibmi_group_by_coverage)
    )
    return apply_value_rules(
        result, "ASLOB", mapping_rules(ibmi, "CoverageCode", rules.ibmi_aslob_by_coverage)
    )


def apply_aslob_corrections(ctx: Context, df: DataFrame) -> DataFrame:
    """Fills missing ASLOB values from reference files and Origami defaults.

    Args:
        ctx: Pipeline context.
        df: Rows with policy and claim data.

    Returns:
        Rows with ASLOB corrected.
    """
    rules = ctx.rules
    result = df

    manual = read_reference_csv(ctx, "aslob_manual")
    if manual is not None:
        keys = ["PolicySource", "CoverageCode", "CoverageExtension"]
        require_columns(manual, keys + ["Correct ASLOB"], "ASLOB manual update file")
        lookup = prepare_lookup(
            ctx, manual.select(*keys, F.col("Correct ASLOB").alias("_aslob_new")), keys,
            "ASLOB manual update file",
        )
        result = (
            result.join(F.broadcast(lookup), keys, "left")
            .withColumn(
                "ASLOB",
                F.when(F.col("ASLOB").isNull(), F.col("_aslob_new")).otherwise(F.col("ASLOB")),
            )
            .select(*df.columns)
        )

    fallback = [
        {"when": {"ASLOB": None, "PolicySource": rules.origami_source, "CoverageCode": code},
         "value": value}
        for code, value in rules.origami_aslob_fallback.items()
    ]
    result = apply_value_rules(result, "ASLOB", fallback)

    ba_plus = read_reference_csv(ctx, "ba_plus_aslob")
    if ba_plus is not None:
        keys = ["PolicyNumber", "PolicyVersion", "CoverageCode"]
        require_columns(ba_plus, keys, "BA PLUS ASLOB file")
        lookup = ba_plus.select(*keys).dropDuplicates().withColumn("_ba_plus_match", F.lit(1))
        columns = result.columns
        result = (
            result.join(F.broadcast(lookup), keys, "left")
            .withColumn(
                "ASLOB",
                F.when(
                    (F.col("_ba_plus_match") == 1) & F.col("ClaimNumber").isNull(),
                    F.lit(rules.ba_plus_aslob_value),
                ).otherwise(F.col("ASLOB")),
            )
            .select(*columns)
        )
    return result


def apply_zip_corrections(ctx: Context, df: DataFrame) -> DataFrame:
    """Fills and corrects zip codes from reference files and fixed overrides.

    Args:
        ctx: Pipeline context.
        df: Rows with ZipCode.

    Returns:
        Rows with ZipCode corrected.
    """
    result = df
    columns = df.columns

    missing = read_reference_csv(ctx, "missing_zip")
    if missing is not None:
        require_columns(missing, ["InTransactionId", "ZIP Code"], "missing zip file")
        lookup = prepare_lookup(
            ctx,
            missing.select("InTransactionId", F.col("ZIP Code").alias("_zip_manual")),
            ["InTransactionId"], "missing zip file",
        )
        result = (
            result.join(F.broadcast(lookup), "InTransactionId", "left")
            .withColumn("ZipCode", F.coalesce(F.col("ZipCode"), F.col("_zip_manual")))
            .select(*columns)
        )

    result = apply_value_rules(result, "ZipCode", ctx.rules.zip_overrides)

    fixes = read_reference_csv(ctx, "zip_fix")
    if fixes is not None:
        keys = ["InTransactionId", "CoverageId"]
        require_columns(fixes, keys + ["ZIP Code"], "zip fix file")
        lookup = prepare_lookup(
            ctx, fixes.select(*keys, F.col("ZIP Code").alias("_zip_fix")), keys, "zip fix file"
        )
        result = (
            result.join(F.broadcast(lookup), keys, "left")
            .withColumn("ZipCode", F.coalesce(F.col("_zip_fix"), F.col("ZipCode")))
            .select(*columns)
        )
    return result


def assign_territory_codes(ctx: Context, df: DataFrame) -> DataFrame:
    """Adds TerritoryCode from the territory file for the reporting states.

    Args:
        ctx: Pipeline context.
        df: Rows with ZipCode and PrimaryRiskState.

    Returns:
        Rows with TerritoryCode appended.
    """
    states = list(ctx.cfg.reporting_states)
    territory = read_reference_csv(ctx, "territory")
    require_columns(territory, ["ZIP", "StateAbbrev", "Territory"], "territory file")
    lookup = prepare_lookup(
        ctx,
        territory.filter(F.col("StateAbbrev").isin(*states)).select(
            F.col("ZIP").alias("_t_zip"),
            F.col("StateAbbrev").alias("_t_state"),
            F.col("Territory").alias("_t_territory"),
        ),
        ["_t_zip", "_t_state"], "territory file",
    )
    joined = df.join(
        F.broadcast(lookup),
        (F.col("ZipCode") == F.col("_t_zip")) & (F.col("PrimaryRiskState") == F.col("_t_state")),
        "left",
    )
    return joined.select(
        *df.columns,
        F.when(F.col("PrimaryRiskState").isin(*states), F.col("_t_territory"))
        .otherwise(F.lit(None)).alias("TerritoryCode"),
    )


def derive_pip_indicator(ctx: Context, df: DataFrame) -> DataFrame:
    """Derives HasPIPVUWCCoveredInd from the rating basis of PIP coverages.

    Args:
        ctx: Pipeline context.
        df: Rows with policy and claim data.

    Returns:
        Rows with HasPIPVUWCCoveredInd appended.
    """
    rules = ctx.rules
    eligible = (
        df.filter(
            (F.col("PolicySource") == rules.ibmi_source)
            & (F.col("PrimaryRiskState") == rules.has_pip_state)
            & F.col("CoverageExtension").isin(*rules.has_pip_extensions)
            & F.col("RiskItemId").isNotNull()
        )
        .select("InTransactionId", "RiskItemId")
        .distinct()
    )
    rating_basis = (
        restrict_to_keys(ctx.read_ods("question_answer"), eligible, ["InTransactionId"])
        .filter(F.col("QuestionCode") == rules.rating_basis_question_code)
        .select("InTransactionId", "RiskItemId", F.col("TextValue").cast("int").alias("RB_TextValue"))
    )
    valid = eligible.join(rating_basis, ["InTransactionId", "RiskItemId"], "inner")
    with_basis = df.join(valid, ["InTransactionId", "RiskItemId"], "left").select(
        *df.columns, "RB_TextValue"
    )
    with_indicator = with_basis.withColumn(
        "HasPIPVUWCCoveredInd", when_chain(rules.has_pip_rules, F.lit(None))
    )
    return with_indicator.drop("RB_TextValue")


def apply_pip_overrides(ctx: Context, df: DataFrame) -> DataFrame:
    """Applies manual HasPIPVUWCCoveredInd values from the reference file.

    Args:
        ctx: Pipeline context.
        df: Rows with HasPIPVUWCCoveredInd.

    Returns:
        Rows with the manual values applied.
    """
    manual = read_reference_csv(ctx, "has_pip")
    if manual is None:
        return df
    keys = ["PolicyNumber", "PolicyVersion", "InTransactionId", "RiskItemId", "CoverageExtension"]
    require_columns(manual, keys + ["HasPIPVUWCCoveredInd"], "PIP indicator file")
    lookup = prepare_lookup(
        ctx,
        manual.select(*keys, F.col("HasPIPVUWCCoveredInd").alias("_pip_manual")),
        keys, "PIP indicator file",
    )
    return (
        df.join(F.broadcast(lookup), keys, "left")
        .withColumn(
            "HasPIPVUWCCoveredInd",
            F.coalesce(F.col("_pip_manual"), F.col("HasPIPVUWCCoveredInd")),
        )
        .select(*df.columns)
    )


def apply_claim_code_rules(ctx: Context, df: DataFrame) -> DataFrame:
    """Normalizes coverage extension, claim code and option values.

    Args:
        ctx: Pipeline context.
        df: Rows with policy and claim data.

    Returns:
        Rows with CoverageExtension, ClaimCode, Option1 and HasPIPVUWCCoveredInd updated.
    """
    rules = ctx.rules
    result = apply_value_rules(df, "CoverageExtension", rules.post_extension_rules)
    result = result.withColumn(
        "ClaimCode",
        F.when(
            F.col("ParentCoverageId").isNull() & F.col("ClaimCode").isNull(),
            F.concat_ws(rules.claim_code_separator, F.col("CoverageCode"), F.col("CoverageExtension")),
        ).otherwise(F.col("ClaimCode")),
    )
    result = apply_value_rules(result, "Option1", rules.option1_rules)
    pair = F.concat_ws("_", F.col("CoverageCode"), F.col("CoverageExtension"))
    return result.withColumn(
        "HasPIPVUWCCoveredInd",
        F.when(
            pair.isin(*rules.has_pip_default_pairs) & F.col("HasPIPVUWCCoveredInd").isNull(),
            F.lit(0),
        ).otherwise(F.col("HasPIPVUWCCoveredInd")),
    )


def map_niss_codes(ctx: Context, df: DataFrame) -> DataFrame:
    """Adds the NISS coverage, subline, type of loss and class codes.

    Rows are matched to the NISS mapping file on claim code, state, option,
    deductible and workers compensation indicator. Rows without a claim code
    are not part of the NISS output.

    Args:
        ctx: Pipeline context.
        df: Rows with policy and claim data.

    Returns:
        Rows with NISS_Coverage_Code, NISS_Subline_Code, NISS_Type_of_Loss and
        NISS_Class_Codes appended.
    """
    rules = ctx.rules
    mapping = read_reference_csv(ctx, "niss_mapping")
    source_columns = [
        "ODS Child Coverage Code", "Coverage Code", "State", "Coverage Extension", "Deductible",
        "Deductible Applies to Option1", "Covered By Worker's Compensation", "ClaimCode",
        "NISS Coverage Code", "NISS Subline Code (1)", "NISS Type of Loss (1)",
        "NISS Class Codes (6)",
    ]
    require_columns(mapping, source_columns, "NISS mapping file")
    lookup = (
        mapping
        .withColumn("_deductible", F.when(F.col("Deductible") == 0, None).otherwise(F.col("Deductible")))
        .select(
            F.col("ClaimCode").alias("_m_claim_code"),
            F.col("State").alias("_m_state"),
            F.col("Deductible Applies to Option1").alias("_m_option"),
            F.col("_deductible").alias("_m_deductible"),
            F.col("Covered By Worker's Compensation").alias("_m_worker"),
            F.coalesce(F.col("ODS Child Coverage Code"), F.col("Coverage Code")).alias("_m_effective_code"),
            F.col("Coverage Extension").alias("_m_extension"),
            F.lpad(F.col("NISS Coverage Code").cast("string"), 3, "0").alias("NISS_Coverage_Code"),
            F.col("NISS Subline Code (1)").alias("NISS_Subline_Code"),
            F.col("NISS Type of Loss (1)").alias("NISS_Type_of_Loss"),
            F.col("NISS Class Codes (6)").alias("NISS_Class_Codes"),
        )
        .dropDuplicates([
            "_m_effective_code", "_m_state", "_m_extension", "_m_deductible",
            "_m_option", "_m_worker", "_m_claim_code",
        ])
    )

    claim_code = F.upper(F.trim(F.col("ClaimCode")))
    state = F.upper(F.trim(F.col("PrimaryRiskState")))
    keep_deductible = reduce(
        lambda left, right: left | right,
        [claim_code == code for code in rules.deductible_claim_codes]
        + [
            (claim_code == item["claim_code"]) & (state == item["state"])
            for item in rules.deductible_state_claim_codes
        ],
    )
    prepared = (
        df
        .withColumn(
            "_option_clean",
            F.when(F.col("Option1").isin(*rules.valid_option1_values), F.col("Option1")).otherwise(None),
        )
        .withColumn(
            "Deductible1",
            F.when(~keep_deductible, None).otherwise(F.col("Deductible1")),
        )
        .withColumn(
            "_deductible_clean",
            F.when(F.col("Deductible1") == 0.0, None).otherwise(F.col("Deductible1")),
        )
    )
    # Only rows that carry a claim code are mapped. Rows without one are dropped here,
    # which matches the legacy notebook.
    with_claim_code = prepared.filter(F.col("ClaimCode").isNotNull())
    matched = with_claim_code.join(
        F.broadcast(lookup),
        (F.col("ClaimCode") == F.col("_m_claim_code"))
        & (F.col("PrimaryRiskState") == F.col("_m_state"))
        & null_safe_equal("_option_clean", "_m_option")
        & null_safe_equal("_deductible_clean", "_m_deductible")
        & F.col("HasPIPVUWCCoveredInd").cast("int").eqNullSafe(F.col("_m_worker").cast("int")),
        "left",
    )
    return matched.select(
        *df.columns, "NISS_Coverage_Code", "NISS_Subline_Code", "NISS_Type_of_Loss", "NISS_Class_Codes"
    )


def refine_niss_class_codes(ctx: Context, df: DataFrame) -> DataFrame:
    """Applies NISS code overrides and resolves the class code.

    Args:
        ctx: Pipeline context.
        df: Rows with NISS codes.

    Returns:
        Rows with final NISS_Class_Codes and overrides applied.
    """
    rules = ctx.rules
    result = apply_multi_column_rules(df, rules.niss_code_overrides)
    result = result.withColumn(
        "NISS_Class_Codes",
        F.when(F.col("NISS_Class_Codes") == rules.vehicle_class_placeholder, F.col("ClassCode"))
        .otherwise(F.col("NISS_Class_Codes")),
    )
    result = result.withColumn(
        "NISS_Class_Codes",
        F.substring(F.col("NISS_Class_Codes"), 1, rules.niss_class_code_length),
    )
    corrections = read_reference_csv(ctx, "corrected_class")
    if corrections is None:
        return result
    require_columns(corrections, ["Class", "NISS Code"], "corrected class file")
    lookup = prepare_lookup(
        ctx,
        corrections.select(F.col("Class").alias("_cc_class"), F.col("NISS Code").alias("_cc_code")),
        ["_cc_class"], "corrected class file",
    )
    columns = result.columns
    return (
        result.join(F.broadcast(lookup), F.col("NISS_Class_Codes") == F.col("_cc_class"), "left")
        .withColumn(
            "NISS_Class_Codes",
            F.when(F.col("_cc_code").isNotNull(), F.col("_cc_code")).otherwise(F.col("NISS_Class_Codes")),
        )
        .select(*columns)
    )


def apply_cause_of_loss(ctx: Context, df: DataFrame) -> DataFrame:
    """Fills the NISS type of loss of claim rows from the cause of loss.

    Args:
        ctx: Pipeline context.
        df: Rows with NISS codes.

    Returns:
        Rows with NISS_Type_of_Loss set for claim rows and cleared for premium rows.
    """
    mapping = read_reference_csv(ctx, "cause_of_loss")
    require_columns(mapping, ["Numeric Code", "Cause Of Loss Code3"], "cause of loss file")
    lookup = prepare_lookup(
        ctx,
        mapping.select(
            F.col("Numeric Code").alias("_col_code"),
            F.col("Cause Of Loss Code3").alias("_col_type"),
        ),
        ["_col_code"], "cause of loss file",
    )
    columns = df.columns
    filled = (
        df.join(F.broadcast(lookup), F.col("CauseOfLoss") == F.col("_col_code"), "left")
        .withColumn(
            "NISS_Type_of_Loss",
            F.when(
                F.col("ClaimNumber").isNotNull()
                & F.col("NISS_Type_of_Loss").isNull()
                & F.col("_col_type").isNotNull(),
                F.col("_col_type"),
            ).otherwise(F.col("NISS_Type_of_Loss")),
        )
        .select(*columns)
    )
    return filled.withColumn(
        "NISS_Type_of_Loss",
        F.when(F.col("ClaimNumber").isNull(), F.lit(None)).otherwise(F.col("NISS_Type_of_Loss")),
    )


def _limit_column(code: str, limit: str) -> str:
    """Builds a safe column name for a coverage limit, for example MED_EXP_Limit1."""
    return re.sub(r"\W+", "_", f"{code}_{limit}")


def derive_exception_codes(ctx: Context, df: DataFrame) -> DataFrame:
    """Derives NISS_Exception_Code from the limits of Pennsylvania PIP coverages.

    The limits of medical expense, accidental death, funeral expense and income
    loss are read for each risk item and matched to the exception mapping file.
    Mapping cells with Any or All Other match every limit.

    Args:
        ctx: Pipeline context.
        df: Rows with policy and claim data.

    Returns:
        Rows with NISS_Exception_Code appended.
    """
    rules = ctx.rules
    all_of = [(code, limit) for code, limit in rules.exception_all_of]
    any_of = [(code, limit) for code, limit in rules.exception_any_of]
    pairs = all_of + any_of
    codes = list(dict.fromkeys(code for code, _ in pairs))

    anchors = (
        df.filter(
            (F.col("CoverageCode") == rules.exception_anchor_coverage)
            & F.col("CoverageExtension").isin(*rules.exception_anchor_extensions)
            & (F.col("PrimaryRiskState") == rules.exception_anchor_state)
        )
        .select("InTransactionId", "RiskItemId")
        .dropDuplicates()
    )
    limits = (
        restrict_to_keys(ctx.read_ods("coverage"), anchors, ["InTransactionId"])
        .join(anchors, ["InTransactionId", "RiskItemId"], "inner")
        .filter(F.col("CoverageCode").isin(*codes))
        .select("InTransactionId", "RiskItemId", "CoverageCode", "Limit1", "Limit2")
    )
    pivoted = (
        limits.groupBy("InTransactionId", "RiskItemId")
        .pivot("CoverageCode", codes)
        .agg(F.first("Limit1").alias("Limit1"), F.first("Limit2").alias("Limit2"))
    )
    source = pivoted.select(
        "InTransactionId", "RiskItemId",
        *[
            F.col(f"`{code}_{limit}`").cast("int").alias(f"s_{_limit_column(code, limit)}")
            for code, limit in pairs
        ],
    )

    mapping = read_reference_csv(ctx, "exception_mapping")
    header_names = [f"{code} {limit}" for code, limit in pairs]
    require_columns(mapping, header_names + ["NISS Exception Code"], "exception mapping file")
    normalized = []
    for (code, limit), header in zip(pairs, header_names):
        raw = F.col(f"`{header}`")
        normalized.append(
            F.when(raw == "Any", "Any").when(raw == "All Other", "AO")
            .otherwise(raw.cast("int")).alias(f"m_{_limit_column(code, limit)}")
        )
    exception_map = mapping.select(
        *normalized, F.col("NISS Exception Code").alias("_m_exception_code")
    )

    def matches(code: str, limit: str) -> Column:
        """Tells whether a limit matches the mapping cell, including wildcards."""
        name = _limit_column(code, limit)
        source_value = F.col(f"s_{name}")
        mapped = F.col(f"m_{name}")
        return (
            (mapped == "Any") | (mapped == "AO")
            | (mapped == source_value.cast("string"))
            | (mapped.cast("int") == source_value)
        )

    condition = reduce(
        lambda left, right: left & right, [matches(c, l) for c, l in all_of], F.lit(True)
    )
    if any_of:
        condition = condition & reduce(
            lambda left, right: left | right, [matches(c, l) for c, l in any_of]
        )
    matched = (
        source.join(F.broadcast(exception_map), condition, "left")
        .select("InTransactionId", "RiskItemId", F.col("_m_exception_code").alias("NISS_Exception_Code"))
        .dropDuplicates(["InTransactionId", "RiskItemId"])
    )
    return df.join(matched, ["InTransactionId", "RiskItemId"], "left").select(
        *df.columns, "NISS_Exception_Code"
    )


def apply_exception_code_rules(ctx: Context, df: DataFrame) -> DataFrame:
    """Applies limit based exception codes and keeps codes only where they apply.

    Args:
        ctx: Pipeline context.
        df: Rows with NISS_Exception_Code.

    Returns:
        Rows with NISS_Exception_Code updated.
    """
    rules = ctx.rules
    by_coverage: Optional[Column] = None
    for coverage, spec in rules.exception_by_limit.items():
        limit_rules = [
            {"when": {"Limit1": limit}, "value": code} for limit, code in spec["limits"]
        ]
        value = when_chain(limit_rules, F.lit(spec.get("default")))
        is_coverage = F.col("CoverageCode") == coverage
        by_coverage = (
            F.when(is_coverage, value) if by_coverage is None
            else by_coverage.when(is_coverage, value)
        )
    result = df
    if by_coverage is not None:
        result = result.withColumn(
            "NISS_Exception_Code", by_coverage.otherwise(F.col("NISS_Exception_Code"))
        )
    result = result.withColumn(
        "NISS_Exception_Code", F.lpad(F.col("NISS_Exception_Code").cast("string"), 2, "0")
    )
    applicable = F.col("CoverageCode").isin(*rules.exception_applicable_coverages)
    for item in rules.exception_applicable_with_extension:
        applicable = applicable | (
            (F.col("CoverageCode") == item["CoverageCode"])
            & (F.col("CoverageExtension") == item["CoverageExtension"])
        )
    return result.withColumn(
        "NISS_Exception_Code",
        F.when(applicable, F.col("NISS_Exception_Code")).otherwise(None),
    )


def calculate_exposure(ctx: Context, df: DataFrame) -> DataFrame:
    """Calculates the written exposure of every row.

    Origami premium rows without a proration value use the default value. The
    exposure is the proration multiplied by the number of months, rounded half
    up, and negative when the premium is negative.

    Args:
        ctx: Pipeline context.
        df: Rows with ProrataPercent and premium amounts.

    Returns:
        Rows with ProrataPercent updated and Final_Exposure appended.
    """
    rules = ctx.rules
    prorata = F.col("ProrataPercent")
    missing_prorata = prorata.isNull() | (F.trim(prorata) == "") | (prorata == "null")
    result = df.withColumn(
        "ProrataPercent",
        F.when(
            (F.col("PolicySource") == rules.origami_source)
            & F.col("ClaimNumber").isNull()
            & missing_prorata,
            F.lit(rules.prorata_default_value),
        ).otherwise(F.col("ProrataPercent")),
    )
    premium = F.when(F.col("OnsetAmount") != 0, F.col("OnsetAmount")).otherwise(F.col("OffsetAmount"))
    raw = F.col("ProrataPercent") * F.lit(rules.exposure_months)
    rounded = F.when((raw - F.floor(raw)) >= 0.5, F.ceil(raw)).otherwise(F.floor(raw))
    return result.withColumn(
        "Final_Exposure", F.when(premium < 0, -rounded).otherwise(rounded)
    )


def apply_state_codes(ctx: Context, df: DataFrame) -> DataFrame:
    """Replaces state abbreviations with NISS state codes.

    Args:
        ctx: Pipeline context.
        df: Rows with PrimaryRiskState.

    Returns:
        Rows with PrimaryRiskState converted. Unknown states keep their value.
    """
    state_rules = [
        {"when": {"PrimaryRiskState": state}, "value": code}
        for state, code in ctx.rules.state_code_map.items()
    ]
    return apply_value_rules(df, "PrimaryRiskState", state_rules)


def apply_lookup_overrides(ctx: Context, df: DataFrame, key: str, keys: Sequence[str], label: str) -> DataFrame:
    """Overwrites columns with the non-null values of a correction file.

    Every column of the file that is not a key and exists in the data replaces
    the data value when the file value is not null.

    Args:
        ctx: Pipeline context.
        df: Rows to correct.
        key: Key of the file in Rules.reference_files.
        keys: Join key columns.
        label: Name used in messages.

    Returns:
        Corrected rows with the original column order.
    """
    fixes = read_reference_csv(ctx, key)
    if fixes is None:
        return df
    lookup = prepare_lookup(ctx, fixes, keys, label)
    update_columns = [name for name in lookup.columns if name not in keys and name in df.columns]
    if not update_columns:
        return df
    renamed = lookup.select(*keys, *[F.col(name).alias(f"_fix_{name}") for name in update_columns])
    joined = df.join(F.broadcast(renamed), list(keys), "left")
    return joined.select(*[
        F.coalesce(F.col(f"_fix_{name}"), F.col(name)).alias(name) if name in update_columns else F.col(name)
        for name in df.columns
    ])


def apply_final_corrections(ctx: Context, df: DataFrame) -> DataFrame:
    """Applies fixed data corrections to coverage codes and exception codes.

    Args:
        ctx: Pipeline context.
        df: Fully mapped rows.

    Returns:
        Corrected rows.
    """
    rules = ctx.rules
    result = apply_value_rules(df, "CoverageCode", rules.claim_coverage_code_overrides)
    result = apply_lookup_overrides(
        ctx, result, "datafix_offset",
        ["InTransactionId", "PolicyNumber", "PolicyVersion", "PrimaryRiskState",
         "CoverageCode", "OffsetAmount"],
        "offset data fix file",
    )
    result = apply_lookup_overrides(
        ctx, result, "datafix_claims", ["ClaimNumber", "InTransactionId", "CoverageCode"],
        "claim data fix file",
    )
    return apply_value_rules(result, "NISS_Exception_Code", rules.exception_code_overrides)


def transform_rows(ctx: Context) -> DataFrame:
    """Runs the transformation stage and stores the final transformed table.

    Args:
        ctx: Pipeline context.

    Returns:
        The stored table.
    """
    cfg = ctx.cfg
    df = spark.table(ctx.out(cfg.table_onset_offset_address))
    df = apply_coverage_attribute_rules(ctx, df)
    df = apply_aslob_corrections(ctx, df)
    df = apply_zip_corrections(ctx, df)
    df = assign_territory_codes(ctx, df)
    df = derive_pip_indicator(ctx, df)
    df = apply_pip_overrides(ctx, df)
    if cfg.drop_duplicate_rows:
        df = df.dropDuplicates()
    df = apply_claim_code_rules(ctx, df)
    df = map_niss_codes(ctx, df)
    df = refine_niss_class_codes(ctx, df)
    df = apply_cause_of_loss(ctx, df)
    df = derive_exception_codes(ctx, df)
    df = apply_exception_code_rules(ctx, df)
    df = calculate_exposure(ctx, df)
    df = apply_state_codes(ctx, df)
    df = apply_final_corrections(ctx, df)
    return write_delta_table(ctx, df, ctx.out(cfg.table_final_transformed))

## Stage 7: Report files

In [ ]:
EXPECTED_FIELD_COUNT = 55
CLAIM_INDICATOR_COLUMN = "Class Codes - first 3 characters"


def load_layout_constants(ctx: Context) -> Dict[str, str]:
    """Returns the constant values of the NISS layout.

    The parameter values are used first. When constants_table is set, its values
    replace the parameter values for the same column names.

    Args:
        ctx: Pipeline context.

    Returns:
        Mapping of constant name to value.
    """
    constants = dict(ctx.cfg.layout_constants)
    table = ctx.cfg.constants_table
    if not table:
        return constants
    rows = (
        spark.table(table)
        .groupBy("ColumnName")
        .agg(F.max("ConstantValue").alias("ConstantValue"))
        .collect()
    )
    for row in rows:
        if row["ColumnName"] in constants:
            constants[row["ColumnName"]] = str(row["ConstantValue"])
    logger.info("Layout constants: %s", constants)
    return constants


def niss_layout(cfg: RunConfig, constants: Dict[str, str]) -> List[Tuple[str, Column]]:
    """Defines the 55 fields of the NISS record in file order.

    Blank fields are null so that they are written as empty values.

    Args:
        cfg: Run configuration.
        constants: Layout constants.

    Returns:
        List of (field name, expression) in record order.
    """
    blank = F.lit(None).cast("string")

    def text(name: str) -> Column:
        """Casts a column to string."""
        return F.col(name).cast("string")

    written_premium = (
        F.when(F.col("OnsetAmount").isNotNull() & (F.col("OnsetAmount") != 0), F.col("OnsetAmount"))
        .when(F.col("OffsetAmount").isNotNull() & (F.col("OffsetAmount") != 0), F.col("OffsetAmount"))
    )
    premium_id = F.when(
        F.col("ClaimNumber").isNotNull(),
        F.concat_ws("_", text("ClaimNumber"), text("ClaimantNumber"), text("CoverageCode")),
    ).otherwise(F.concat_ws("_", text("InTransactionId"), text("CoverageId")))

    def rounded_int(column: Column) -> Column:
        """Rounds an amount to a whole number."""
        return F.round(column.cast("double"), 0).cast("int")

    layout = [
        ("Blank_1", blank),
        ("CompanyNumber", F.rpad(F.lit(constants["CompanyNumber"]), 3, " ")),
        ("Blank_5_6", blank),
        ("CalendarYear", F.lpad(F.lit(cfg.two_digit_calendar_year), 2, " ")),
        ("CallYear", F.lpad(F.lit(cfg.two_digit_call_year), 2, "0")),
        ("StateCode", F.rpad(text("PrimaryRiskState"), 2, " ")),
        ("LineCode", F.rpad(F.lit(constants["LineCode"]), 1, " ")),
        ("Blank_14_15", blank),
        ("AccidentYear", F.substring(text("AccidentYear"), 3, 2)),
        ("Blank_18_22", blank),
        ("NISSCoverageCode", F.lpad(text("NISS_Coverage_Code"), 3, "0")),
        ("Blank_26_30", blank),
        ("TerritoryCode", F.lpad(text("TerritoryCode"), 3, "0")),
        ("RatingZoneCode", blank),
        ("TerminalZoneCode", blank),
        ("NSSZIPCode", blank),
        ("Blank_42_45", blank),
        ("ClassificationCode", F.rpad(text("NISS_Class_Codes"), 6, "0")),
        ("EligibilityPointsCode", blank),
        ("Blank_54_55", blank),
        ("AgeGroupCode", blank),
        ("ManufacturersModelYear", blank),
        ("Blank_59_60", blank),
        ("CommercialIndicatorCode", F.rpad(F.lit(constants["CommercialIndicatorCode"]), 1, " ")),
        ("NISS_Exception_Code", F.lpad(text("NISS_Exception_Code"), 2, "0")),
        ("ForgivenessCode", blank),
        ("ClaimantLevelIndicator", F.rpad(F.lit(constants["ClaimantLevelIndicator"]), 1, " ")),
        ("PassiveRestraintCode", blank),
        ("DefensiveDriverCreditCode", blank),
        ("Blank_69", blank),
        ("AntiTheftDeviceCode", blank),
        ("DaytimeRunningLampsDiscountCode", blank),
        ("Blank_72_74", blank),
        ("PolicyLimitsCode", blank),
        ("DeductibleCode", blank),
        ("Blank_79", blank),
        ("SupplementalSpousalLiabilityCode", blank),
        ("Blank_81", blank),
        ("SublineOfBusinessCode", F.rpad(text("NISS_Subline_Code"), 1, " ")),
        ("Blank_83", blank),
        ("TypeOfLossCode", F.rpad(text("NISS_Type_of_Loss"), 1, " ")),
        ("LiabilityNoFaultCode", blank),
        ("Blank_86_98", blank),
        ("AnnualStatementLineOfBusinessCode", F.rpad(text("ASLOB"), 3, " ")),
        ("Blank_102_103", blank),
        ("WrittenExposure", F.col("Final_Exposure")),
        ("WrittenPremium", F.round(
            F.regexp_replace(written_premium.cast("string"), ",", "").cast("double"), 0
        ).cast("int")),
        ("PaidLosses", rounded_int(F.col("PaidLossAmount"))),
        ("PaidAllocatedLossAdjustmentExpenses", rounded_int(F.col("AllocatedLossAdjExp"))),
        ("OutstandingLossesIncludingALAE", rounded_int(F.col("ReserveAmount"))),
        ("NumberOfPaidClaims", F.col("PaidCount").cast("int")),
        ("NumberOfOutstandingClaims", F.col("ReserveCount").cast("int")),
        ("ReservedForNISSUse", blank),
        ("ClaimNumberPremiumID", F.rpad(premium_id.substr(1, 16), 16, " ")),
        ("ClaimantNumber", F.lpad(text("ClaimantNumber").substr(1, 3), 3, "0")),
    ]
    if len(layout) != EXPECTED_FIELD_COUNT:
        raise ValueError(f"NISS layout must have {EXPECTED_FIELD_COUNT} fields, found {len(layout)}")
    return layout


def apply_layout_adjustments(ctx: Context, df: DataFrame) -> DataFrame:
    """Applies the final field rules that depend on the classification code.

    Args:
        ctx: Pipeline context.
        df: Rows with the NISS layout columns.

    Returns:
        Rows with CommercialIndicatorCode, WrittenExposure and NISS_Exception_Code adjusted.
    """
    rules = ctx.rules
    indicator = read_reference_csv(ctx, "claim_indicator_classes")
    require_columns(indicator, [CLAIM_INDICATOR_COLUMN], "claim indicator class file")
    valid_codes = [
        row[0]
        for row in indicator.select(
            F.lpad(F.col(CLAIM_INDICATOR_COLUMN).cast("string"), 3, "0").alias("class_prefix")
        ).distinct().collect()
    ]
    class_prefix = F.substring(F.col("ClassificationCode"), 1, 3)
    keep_indicator = class_prefix.isin(*valid_codes) if valid_codes else F.lit(False)
    result = df.withColumn(
        "CommercialIndicatorCode",
        F.when(keep_indicator, F.col("CommercialIndicatorCode")).otherwise(None),
    )
    result = result.withColumn(
        "WrittenExposure",
        F.when(F.col("ClassificationCode").isin(*rules.exposure_blank_class_codes), None)
        .otherwise(F.col("WrittenExposure")),
    )
    result = apply_value_rules(result, "NISS_Exception_Code", rules.final_exception_overrides)
    return result.withColumn(
        "NISS_Exception_Code", F.lpad(F.col("NISS_Exception_Code").cast("string"), 2, "0")
    )


def build_transmittal(ctx: Context, df: DataFrame, constants: Dict[str, str]) -> DataFrame:
    """Builds one transmittal record per state with the file totals.

    Args:
        ctx: Pipeline context.
        df: Formatted rows.
        constants: Layout constants.

    Returns:
        Transmittal records.
    """
    blank = F.lit(None).cast("string")
    totals = df.groupBy("PrimaryRiskState").agg(
        F.sum("WrittenExposure").alias("WrittenExposure"),
        F.sum("WrittenPremium").alias("WrittenPremium"),
        F.sum("PaidLosses").alias("PaidLosses"),
        F.sum("PaidAllocatedLossAdjustmentExpenses").alias("PaidALAE"),
        F.sum("OutstandingLossesIncludingALAE").alias("OutstandingLosses"),
        F.sum("NumberOfPaidClaims").alias("NumPaidClaims"),
        F.sum("NumberOfOutstandingClaims").alias("NumOutstandingClaims"),
    )
    return totals.select(
        F.lit("T").alias("TransmittalIdentifierCode"),
        F.lit(constants["CompanyNumber"]).alias("CompanyNumber"),
        blank.alias("Blank_5_8"),
        F.lit(ctx.cfg.two_digit_call_year).alias("CallYear"),
        F.col("PrimaryRiskState").alias("StateCode"),
        F.lit(constants["LineCode"]).alias("LineCode"),
        blank.alias("Blank_14_17"),
        blank.alias("RefileIndicator"),
        blank.alias("Blank_19"),
        blank.alias("NTRIndicator"),
        blank.alias("Blank_21_103"),
        F.col("WrittenExposure").cast("int"),
        F.col("WrittenPremium"),
        F.col("PaidLosses").cast("int"),
        F.col("PaidALAE").cast("int"),
        F.col("OutstandingLosses").cast("int"),
        F.col("NumPaidClaims").cast("int"),
        F.col("NumOutstandingClaims").cast("int"),
    )


def build_summary(ctx: Context, record_count: int, constants: Dict[str, str]) -> DataFrame:
    """Builds the single summary record that holds the data record count.

    Args:
        ctx: Pipeline context.
        record_count: Number of data records in the file.
        constants: Layout constants.

    Returns:
        A one row DataFrame.
    """
    blank = F.lit(None).cast("string")
    return spark.createDataFrame([(1,)], ["dummy"]).select(
        F.lit("S").alias("TransmittalIdentifierCode"),
        F.lit(constants["CompanyNumber"]).alias("CompanyNumber"),
        blank.alias("Blank_1"),
        F.lit(ctx.cfg.two_digit_call_year).alias("CallYear"),
        blank.alias("Blank_2"),
        F.lit(constants["LineCode"]).alias("LineCode"),
        blank.alias("Blank_3"),
        blank.alias("RefileIndicator"),
        blank.alias("Blank_5"),
        F.lit(record_count).alias("RecordCount"),
        blank.alias("Blank_4"),
    )


def run_quality_checks(ctx: Context, df: DataFrame) -> Dict[str, int]:
    """Counts nulls in the required output fields and logs the result.

    Args:
        ctx: Pipeline context.
        df: Final report rows.

    Returns:
        Mapping of check name to count.

    Raises:
        RuntimeError: If strict_quality_gate is on and a required field has nulls.
    """
    required = ctx.rules.quality_required_columns
    aggregates = [F.count(F.lit(1)).alias("total_rows")] + [
        F.sum(F.when(F.col(name).isNull(), 1).otherwise(0)).alias(f"null_{name}")
        for name in required
    ]
    row = df.agg(*aggregates).first()
    results = {key: int(row[key] or 0) for key in row.asDict()}
    for key, value in results.items():
        logger.info("Quality check %-45s %s", key, f"{value:,}")
    failed = {key: value for key, value in results.items() if key.startswith("null_") and value > 0}
    if failed:
        logger.warning("Required fields with null values: %s", failed)
        if ctx.cfg.strict_quality_gate:
            raise RuntimeError(f"Quality gate failed: {failed}")
    return results


def build_report(ctx: Context) -> Dict[str, Any]:
    """Builds the NISS record file, transmittal file, summary file and report table.

    Args:
        ctx: Pipeline context.

    Returns:
        Summary values of the report.
    """
    cfg = ctx.cfg
    constants = load_layout_constants(ctx)
    layout = niss_layout(cfg, constants)
    base = spark.table(ctx.out(cfg.table_final_transformed))
    formatted = apply_layout_adjustments(ctx, base.withColumns(dict(layout)))
    formatted = ctx.cache.keep(formatted, "formatted_rows")

    write_csv_folder(
        build_transmittal(ctx, formatted, constants), ctx.output_path(cfg.output_transmittal), False
    )
    record_count = formatted.count()
    write_csv_folder(
        build_summary(ctx, record_count, constants), ctx.output_path(cfg.output_summary), False
    )
    write_csv_folder(formatted, ctx.output_path(cfg.output_pre_final), True)

    report = formatted.select([name for name, _ in layout])
    write_delta_table(ctx, report, ctx.out(cfg.table_niss_final_report))
    write_csv_folder(report, ctx.output_path(cfg.output_niss_file), False)
    quality = run_quality_checks(ctx, report) if cfg.enable_validations else {}
    return {"record_count": record_count, "quality": quality}

## Run

In [ ]:
def run_pipeline(ctx: Context) -> Dict[str, Any]:
    """Runs every stage of the NISS report in order.

    Intermediate results are stored in Delta tables so that a failed run can be
    investigated from the last stored table. Cached DataFrames are always released.

    Args:
        ctx: Pipeline context.

    Returns:
        Summary values of the report stage.
    """
    report_summary: Dict[str, Any] = {}
    try:
        with stage("1 Policy extract"):
            policy_rows = build_policy_rows(ctx)
        with stage("2 Onset and offset rows"):
            build_onset_and_offset_table(ctx, policy_rows)
        ctx.cache.release_all()
        with stage("3 Claim valuation"):
            claims = prepare_claims(ctx)
        with stage("4 Claim to policy mapping"):
            build_claim_policy_table(ctx, claims)
        ctx.cache.release_all()
        with stage("5 Zip codes"):
            build_address_table(ctx)
        with stage("6 Transformations"):
            transform_rows(ctx)
        with stage("7 Report files"):
            report_summary = build_report(ctx)
    finally:
        ctx.cache.release_all()
    return report_summary


run_started = time.time()
rules = load_rules(rules_override_path.strip())
config = build_run_config(rules)
context = Context(cfg=config, rules=rules, cache=CacheRegistry(config.persist_intermediates))
logger.info(
    "Run settings: calendar_year=%s call_year=%s liability_valuation=%s property_valuation=%s",
    config.calendar_year, config.call_year,
    config.liability.valuation_month, config.property_segment.valuation_month,
)

result = run_pipeline(context)

run_summary = {
    "status": "succeeded",
    "calendar_year": config.calendar_year,
    "call_year": config.call_year,
    "record_count": result.get("record_count"),
    "quality": result.get("quality"),
    "stages": RUN_METRICS,
    "total_seconds": round(time.time() - run_started, 1),
}
logger.info("Run summary:\n%s", json.dumps(run_summary, indent=2, default=str))

# Return the summary to a calling pipeline when the notebook runs inside Fabric.
if "notebookutils" in globals():
    notebookutils.notebook.exit(json.dumps(run_summary, default=str))